# SENTRIX — Real-Data Production Model Retraining V3

**Notebook:** `G:\Sentrix\training\SENTRIX_REAL_DATA_RETRAINING_V3.ipynb`

## Purpose

Train production-oriented SENTRIX AI models using **ONLY real datasets**.

## IMPORTANT

- No synthetic training data
- No synthetic labels
- No synthetic TCI values
- No overwriting of previous SENTRIX models
- All previous models and training runs are preserved
- New models use **V3_REAL** naming
- Training data is located at `G:\Capstone\data`
- SENTRIX project root is `G:\Sentrix`

## Module map

| Module | Task | Architecture | Output |
|---|---|---|---|
| 1 | Violence (fight / nofight) | ResNet18 | `violence_classifier_real_v3.pt` |
| 2 | Audio (5 classes) | Log-Mel CNN | `audio_classifier_real_v3.pt` |
| 3 | Weapon (knife / long_gun / pistol) | YOLOv8 | `weapon_detector_real_v3.pt` |
| 4 | Fire / Smoke | YOLOv8 | `fire_smoke_detector_real_v3.pt` |
| 5 | Multimodal fusion → TCI | XGBoost | `tci_xgboost_real_v3.json` *(only if a real labelled event dataset exists)* |

## What happened to the ANOMALY module

It moved. Anomaly detection is now event-based and lives in its own notebook:

```text
G:\Sentrix\training\SENTRIX_UCF_CRIME_DVS_TRAINING.ipynb
   dataset : G:\event_frame_duration533326\event_frame_duration533326  (.npz event data)
   output  : G:\Sentrix\backend\models\v2_real\ucf_crime_dvs\ucf_crime_dvs_anomaly_v1.pt
```

The old image-based `anomaly_data` pipeline is **not** carried into V3. The UCF-Crime-DVS
model supplies the `anomaly_score` that MODULE 5 (fusion) consumes.

## Execution order

Run the cells top to bottom. Modules 1–4 are independent of each other — if you only want to
retrain one model, run **CELL 2 → CELL 7c** (configuration + helpers) first, then jump to that module.
Module 5 depends on nothing but a real event-level dataset; if none exists it aborts by design.

---
# CELL 2 — Master configuration

Single source of truth for every path, model name and run directory used by this notebook.
Nothing below this cell hard-codes a path.

In [1]:
from pathlib import Path
import os
import sys
import json
import math
import random
import shutil
import time
import hashlib
import warnings
import platform
from datetime import datetime

warnings.filterwarnings("ignore")

# ============================================================
# SENTRIX MASTER PATH CONFIGURATION
# ============================================================
PROJECT_ROOT = Path(r"G:\Sentrix")
DATA_ROOT    = Path(r"G:\Capstone\data")

BACKEND_DIR  = PROJECT_ROOT / "backend"
MODELS_DIR   = BACKEND_DIR / "models"
TRAINING_DIR = PROJECT_ROOT / "training"

# NEW V3 training outputs
V3_RUNS_DIR   = TRAINING_DIR / "runs_v3_real"
V3_MODELS_DIR = MODELS_DIR / "v3_real"
V3_RUNS_DIR.mkdir(parents=True, exist_ok=True)
V3_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# aliases so snippets written against the V2 notebook keep working
V2_RUNS_DIR   = V3_RUNS_DIR
V2_MODELS_DIR = V3_MODELS_DIR

# ============================================================
# DATASET DIRECTORIES
# ============================================================
VIOLENCE_DATA   = DATA_ROOT / "violence_data"
AUDIO_DATA      = DATA_ROOT / "audio_data" / "audioset_clips"
FIRE_SMOKE_DATA = DATA_ROOT / "fire_smoke_data"
WEAPON_DATA     = DATA_ROOT / "weapon_data"

# ============================================================
# ANOMALY — handled by the separate event-based notebook.
# Listed here for reference only; nothing in V3 reads it.
# ============================================================
UCF_DVS_ROOT      = Path(r"G:\event_frame_duration533326\event_frame_duration533326")
UCF_DVS_MODEL     = MODELS_DIR / "v2_real" / "ucf_crime_dvs" / "ucf_crime_dvs_anomaly_v1.pt"
UCF_DVS_NOTEBOOK  = TRAINING_DIR / "SENTRIX_UCF_CRIME_DVS_TRAINING.ipynb"

# ============================================================
# OLD MODEL DIRECTORY — READ ONLY
# ============================================================
OLD_MODELS_DIR = PROJECT_ROOT / "backend" / "models"

# ============================================================
# NEW MODEL NAMES
# ============================================================
VIOLENCE_MODEL_V3   = V3_MODELS_DIR / "violence_classifier_real_v3.pt"
AUDIO_MODEL_V3      = V3_MODELS_DIR / "audio_classifier_real_v3.pt"
WEAPON_MODEL_V3     = V3_MODELS_DIR / "weapon_detector_real_v3.pt"
FIRE_SMOKE_MODEL_V3 = V3_MODELS_DIR / "fire_smoke_detector_real_v3.pt"
FUSION_MODEL_V3     = V3_MODELS_DIR / "tci_xgboost_real_v3.json"

# ============================================================
# GLOBAL RUN SETTINGS
# ============================================================
SEED = 42
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
NUM_WORKERS = 0 if platform.system() == "Windows" else 4   # Windows + notebooks: keep 0

# Shared result registries — filled in by each module, consumed by CELL 27 / 30 / 32
RESULTS  = {}   # module -> metrics dict
ARTIFACTS = {}  # module -> {"model": path, "metadata": path, "run_dir": path}
MODULE_STATUS = {}  # module -> "PENDING" | "TRAINED" | "SKIPPED" | "FAILED"

for _m in ["violence", "audio", "weapon", "fire_smoke", "fusion"]:
    MODULE_STATUS[_m] = "PENDING"

# ============================================================
# OLD-MODEL IMMUTABILITY SNAPSHOT
# Fingerprint every pre-existing model file so CELL 31 can prove
# this notebook did not touch any of them.
# ============================================================
def _fingerprint(path: Path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read(4 * 1024 * 1024))   # first 4 MB is enough to detect a rewrite
    st = path.stat()
    return {"size": st.st_size, "mtime": round(st.st_mtime, 3), "sha256_head": h.hexdigest()}


OLD_MODEL_SNAPSHOT = {}
if OLD_MODELS_DIR.exists():
    for _p in sorted(OLD_MODELS_DIR.iterdir()):
        if _p.is_file() and _p.suffix.lower() in (".pt", ".pth", ".onnx", ".json", ".pkl", ".joblib"):
            try:
                OLD_MODEL_SNAPSHOT[_p.name] = _fingerprint(_p)
            except Exception as e:
                OLD_MODEL_SNAPSHOT[_p.name] = {"error": str(e)}

SNAPSHOT_FILE = V2_RUNS_DIR / f"old_models_snapshot_{RUN_ID}.json"
SNAPSHOT_FILE.write_text(json.dumps(OLD_MODEL_SNAPSHOT, indent=2), encoding="utf-8")

print("=" * 60)
print("SENTRIX V3 REAL-DATA RETRAINING — CONFIGURATION")
print("=" * 60)
print("RUN_ID       :", RUN_ID)
print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_ROOT    :", DATA_ROOT)
print("MODELS_DIR   :", MODELS_DIR)
print("V3_MODELS    :", V3_MODELS_DIR)
print("V3_RUNS      :", V3_RUNS_DIR)
print("AUDIO_DATA   :", AUDIO_DATA)
print()
print("ANOMALY is trained separately (event data):")
print("   notebook :", UCF_DVS_NOTEBOOK.name)
print("   dataset  :", UCF_DVS_ROOT)
print("   model    :", UCF_DVS_MODEL)
print("SEED         :", SEED)
print("NUM_WORKERS  :", NUM_WORKERS)
print()
print(f"Old models fingerprinted (read-only): {len(OLD_MODEL_SNAPSHOT)}")
for _n in OLD_MODEL_SNAPSHOT:
    print("   -", _n)
print("Snapshot saved to:", SNAPSHOT_FILE)

SENTRIX V3 REAL-DATA RETRAINING — CONFIGURATION
RUN_ID       : 20260824_193729
PROJECT_ROOT : G:\Sentrix
DATA_ROOT    : G:\Capstone\data
MODELS_DIR   : G:\Sentrix\backend\models
V3_MODELS    : G:\Sentrix\backend\models\v3_real
V3_RUNS      : G:\Sentrix\training\runs_v3_real
AUDIO_DATA   : G:\Capstone\data\audio_data\audioset_clips

ANOMALY is trained separately (event data):
   notebook : SENTRIX_UCF_CRIME_DVS_TRAINING.ipynb
   dataset  : G:\event_frame_duration533326\event_frame_duration533326
   model    : G:\Sentrix\backend\models\v2_real\ucf_crime_dvs\ucf_crime_dvs_anomaly_v1.pt
SEED         : 42
NUM_WORKERS  : 0

Old models fingerprinted (read-only): 7
   - anomaly_classifier.pt
   - audio_classifier.pt
   - fire_smoke_detector.pt
   - tci_xgboost.json
   - violence_classifier.pt
   - weapon_detector.pt
   - yolov8n.pt
Snapshot saved to: G:\Sentrix\training\runs_v3_real\old_models_snapshot_20260824_193729.json


---
# CELL 3 — Verify all paths

The notebook **STOPS** here if a critical path is missing. Nothing is created,
downloaded or generated to paper over a missing dataset.

In [2]:
paths_to_check = {
    "Project Root":       PROJECT_ROOT,
    "Data Root":          DATA_ROOT,
    "Violence Dataset":   VIOLENCE_DATA,
    "Audio Dataset":      AUDIO_DATA,
    "Fire/Smoke Dataset": FIRE_SMOKE_DATA,
    "Weapon Dataset":     WEAPON_DATA,
    "Backend Models":     MODELS_DIR,
}

# Missing any of these => hard stop.
CRITICAL = {"Project Root", "Data Root", "Backend Models"}

missing = []
print("=" * 60)
print("PATH VERIFICATION")
print("=" * 60)
for name, path in paths_to_check.items():
    exists = path.exists()
    print(f"{name:20} -> {path}")
    print(f"{'':20}    EXISTS: {exists}")
    if not exists:
        missing.append(name)

AVAILABLE_MODULES = {
    "violence":   VIOLENCE_DATA.exists(),
    "audio":      AUDIO_DATA.exists(),
    "weapon":     WEAPON_DATA.exists(),
    "fire_smoke": FIRE_SMOKE_DATA.exists(),
}

print()
print("Trainable modules based on data presence:")
for k, v in AVAILABLE_MODULES.items():
    print(f"   {k:12} : {'YES' if v else 'NO  (dataset directory missing - module will be skipped)'}")

critical_missing = [m for m in missing if m in CRITICAL]
if critical_missing:
    raise RuntimeError(
        "CRITICAL PATHS MISSING: " + ", ".join(critical_missing) +
        "\nFix the paths in CELL 2 before continuing. "
        "This notebook will not create or synthesise data to work around a missing path."
    )

if missing:
    print()
    print("WARNING: non-critical datasets missing:", ", ".join(missing))
    print("Those modules will be SKIPPED (not synthesised).")
else:
    print()
    print("All paths verified.")

print()
print("ANOMALY module: not part of V3 - see", UCF_DVS_NOTEBOOK.name)
print("   event dataset present:", UCF_DVS_ROOT.exists())
print("   event model trained  :", UCF_DVS_MODEL.exists())

PATH VERIFICATION
Project Root         -> G:\Sentrix
                        EXISTS: True
Data Root            -> G:\Capstone\data
                        EXISTS: True
Violence Dataset     -> G:\Capstone\data\violence_data
                        EXISTS: True
Audio Dataset        -> G:\Capstone\data\audio_data\audioset_clips
                        EXISTS: True
Fire/Smoke Dataset   -> G:\Capstone\data\fire_smoke_data
                        EXISTS: True
Weapon Dataset       -> G:\Capstone\data\weapon_data
                        EXISTS: True
Backend Models       -> G:\Sentrix\backend\models
                        EXISTS: True

Trainable modules based on data presence:
   violence     : YES
   audio        : YES
   weapon       : YES
   fire_smoke   : YES

All paths verified.

ANOMALY module: not part of V3 - see SENTRIX_UCF_CRIME_DVS_TRAINING.ipynb
   event dataset present: True
   event model trained  : True


---
# CELL 4 — Environment / GPU check

In [3]:
import importlib

def _ver(mod_name):
    try:
        m = importlib.import_module(mod_name)
        return getattr(m, "__version__", "installed")
    except Exception as e:
        return f"NOT INSTALLED ({type(e).__name__})"

import torch
import torchvision

print("=" * 60)
print("ENVIRONMENT")
print("=" * 60)
print("Python       :", sys.version.split()[0])
print("Platform     :", platform.platform())
print("PyTorch      :", torch.__version__)
print("TorchVision  :", torchvision.__version__)
print("TorchAudio   :", _ver("torchaudio"))
print("OpenCV       :", _ver("cv2"))
print("Scikit-Learn :", _ver("sklearn"))
print("Ultralytics  :", _ver("ultralytics"))
print("XGBoost      :", _ver("xgboost"))
print("Librosa      :", _ver("librosa"))
print("SoundFile    :", _ver("soundfile"))
print("Pandas       :", _ver("pandas"))
print("Pillow       :", _ver("PIL"))
print()

if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("DEVICE:", DEVICE)
if DEVICE == "cuda":
    print("CUDA runtime :", torch.version.cuda)
    print("cuDNN        :", torch.backends.cudnn.version())
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name}  |  {p.total_memory / 1024**3:.1f} GB  |  SM {p.major}.{p.minor}")
    torch.backends.cudnn.benchmark = True
else:
    print("WARNING: training on CPU will be very slow for the YOLO modules.")
    print("See the final cell of this notebook for the accelerated-PyTorch conda setup.")

USE_AMP = (DEVICE == "cuda")
print("Mixed precision (AMP):", USE_AMP)

ENVIRONMENT
Python       : 3.11.15
Platform     : Windows-10-10.0.26200-SP0
PyTorch      : 2.7.1+cu126
TorchVision  : 0.22.1+cu126
TorchAudio   : 2.7.1+cu126
OpenCV       : 4.10.0
Scikit-Learn : 1.5.2
Ultralytics  : 8.3.203
XGBoost      : 2.1.4
Librosa      : 0.10.2.post1
SoundFile    : 0.12.1
Pandas       : 2.2.3
Pillow       : 10.4.0

DEVICE: cuda
CUDA runtime : 12.6
cuDNN        : 90701
GPU 0: NVIDIA GeForce RTX 4060 Laptop GPU  |  8.0 GB  |  SM 8.9
Mixed precision (AMP): True


---
# CELL 5 — Dataset inventory

**NO DATA GENERATION HERE — ONLY COUNTING / INSPECTING EXISTING FILES.**

Expected layout:

```text
anomaly_data      : normal, anomalous
violence_data     : fight, nofight
audio_data        : speech_normal, siren, fire, scream
fire_smoke_data   : fire, smoke            (YOLO)
weapon_data       : knife, long_gun, pistol (YOLO)
```

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
AUDIO_EXTS = {".wav", ".flac", ".mp3", ".ogg", ".m4a"}

# ---- NO DATA GENERATION HERE ----
# ---- ONLY COUNTING / INSPECTING EXISTING FILES ----

def count_files(folder: Path, exts) -> int:
    if not folder.exists():
        return 0
    return sum(1 for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in exts)


def find_class_dirs(root: Path, class_names):
    # Returns {class_name: [dirs...]} searching root/<cls>, root/<split>/<cls>,
    # root/images/<split>/<cls> -- whatever actually exists on disk.
    found = {c: [] for c in class_names}
    if not root.exists():
        return found
    lowered = {c.lower(): c for c in class_names}
    for p in root.rglob("*"):
        if p.is_dir() and p.name.lower() in lowered:
            found[lowered[p.name.lower()]].append(p)
    return found


INVENTORY = {}

def inventory_classification(title, root: Path, class_names, exts):
    print(f"\n{title}")
    print("-" * len(title))
    if not root.exists():
        print("  DIRECTORY MISSING:", root)
        INVENTORY[title] = None
        return
    dirs = find_class_dirs(root, class_names)
    counts = {}
    for c in class_names:
        n = sum(count_files(d, exts) for d in dirs[c])
        counts[c] = n
        where = " | ".join(str(d.relative_to(root)) for d in dirs[c]) or "NOT FOUND"
        print(f"  {c:14}: {n:>7}   [{where}]")
    total = sum(counts.values())
    loose = count_files(root, exts) - total
    if loose > 0:
        print(f"  {'(unlabelled)':14}: {loose:>7}   files not under a known class dir")
    print(f"  {'TOTAL':14}: {total:>7}")
    INVENTORY[title] = counts


def inventory_yolo(title, root: Path, expected_classes):
    print(f"\n{title}")
    print("-" * len(title))
    if not root.exists():
        print("  DIRECTORY MISSING:", root)
        INVENTORY[title] = None
        return
    yaml_files = sorted(root.glob("*.yaml")) + sorted(root.glob("*.yml"))
    print("  data yaml   :", yaml_files[0].name if yaml_files else "NOT FOUND")
    imgs = count_files(root, IMAGE_EXTS)
    labels = sum(1 for p in root.rglob("*.txt") if p.is_file() and p.parent.name.lower() != "")
    print(f"  images      : {imgs}")
    print(f"  label .txt  : {labels}")
    for split in ["train", "valid", "val", "test"]:
        for cand in [root / split, root / "images" / split]:
            if cand.exists():
                print(f"  {split:11} : {count_files(cand, IMAGE_EXTS)} images")
                break
    print("  expected classes:", ", ".join(expected_classes))
    INVENTORY[title] = {"images": imgs, "labels": labels}


print("=" * 60)
print("SENTRIX REAL DATASET INVENTORY")
print("=" * 60)
print("Source:", DATA_ROOT)

inventory_classification("VIOLENCE", VIOLENCE_DATA, ["fight", "nofight"], IMAGE_EXTS)
inventory_classification("AUDIO",    AUDIO_DATA,
                         ["speech_normal", "siren", "fire", "scream"], AUDIO_EXTS)
inventory_yolo("FIRE_SMOKE (YOLO)", FIRE_SMOKE_DATA, ["fire", "smoke"])
inventory_yolo("WEAPON (YOLO)",     WEAPON_DATA,     ["knife", "long_gun", "pistol"])

# CSV manifests present?
print("\nCSV / manifest files found under DATA_ROOT")
print("-" * 42)
for p in sorted(DATA_ROOT.rglob("*.csv")):
    try:
        rel = p.relative_to(DATA_ROOT)
    except ValueError:
        rel = p
    print(f"  {str(rel):55} {p.stat().st_size/1024:8.1f} KB")

(V2_RUNS_DIR / f"inventory_{RUN_ID}.json").write_text(
    json.dumps(INVENTORY, indent=2), encoding="utf-8")
print("\nInventory saved to:", V2_RUNS_DIR / f"inventory_{RUN_ID}.json")

SENTRIX REAL DATASET INVENTORY
Source: G:\Capstone\data

VIOLENCE
--------
  fight         :   20220   [fight]
  nofight       :   19000   [nofight]
  TOTAL         :   39220

AUDIO
-----
  speech_normal :     160   [speech_normal]
  siren         :     156   [siren]
  explosion     :       0   [NOT FOUND]
  fire          :     105   [fire]
  scream        :      55   [scream]
  TOTAL         :     476

FIRE_SMOKE (YOLO)
-----------------
  data yaml   : ._data.yaml
  images      : 24000
  label .txt  : 24000
  train       : 20000 images
  valid       : 4000 images
  expected classes: fire, smoke

WEAPON (YOLO)
-------------
  data yaml   : ._data.yaml
  images      : 18942
  label .txt  : 18946
  train       : 14338 images
  valid       : 3198 images
  test        : 1406 images
  expected classes: knife, long_gun, pistol

CSV / manifest files found under DATA_ROOT
------------------------------------------
  anomaly_data\._train.csv                                     4.0 KB
  anomaly

---
# CELL 6 — Dataset integrity checks

Checks for: missing files, corrupted images, unreadable WAV files, missing YOLO labels,
invalid bounding boxes, duplicate filenames, empty directories, and train/validation leakage.

> Data quality matters more than simply increasing epochs.

In [5]:
from PIL import Image

INTEGRITY = {}

def verify_images(folder: Path, limit=None):
    # Returns (valid, invalid, bad_files)
    valid, invalid, bad = 0, 0, []
    if not folder.exists():
        return 0, 0, []
    files = [f for f in folder.rglob("*") if f.is_file() and f.suffix.lower() in IMAGE_EXTS]
    if limit:
        files = files[:limit]
    for file in files:
        try:
            with Image.open(file) as img:
                img.verify()
            valid += 1
        except Exception:
            invalid += 1
            if len(bad) < 25:
                bad.append(str(file))
    return valid, invalid, bad


def verify_audio(folder: Path, limit=None):
    valid, invalid, bad, durations = 0, 0, [], []
    if not folder.exists():
        return 0, 0, [], []
    try:
        import soundfile as sf
        reader = "soundfile"
    except Exception:
        sf = None
        reader = None
    files = [f for f in folder.rglob("*") if f.is_file() and f.suffix.lower() in AUDIO_EXTS]
    if limit:
        files = files[:limit]
    for file in files:
        try:
            if reader == "soundfile":
                info = sf.info(str(file))
                durations.append(info.duration)
            else:
                import wave
                with wave.open(str(file)) as w:
                    durations.append(w.getnframes() / float(w.getframerate()))
            valid += 1
        except Exception:
            invalid += 1
            if len(bad) < 25:
                bad.append(str(file))
    return valid, invalid, bad, durations


def check_yolo_labels(root: Path):
    # Every image should have a matching .txt; every box must be normalised 0..1
    issues = {"images_without_labels": [], "empty_labels": 0,
              "invalid_boxes": [], "class_ids": {}, "n_images": 0, "n_labels": 0}
    if not root.exists():
        return issues
    imgs = [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
    issues["n_images"] = len(imgs)
    for img in imgs:
        parts = list(img.parts)
        lbl = None
        if "images" in parts:
            i = len(parts) - 1 - parts[::-1].index("images")
            parts[i] = "labels"
            lbl = Path(*parts).with_suffix(".txt")
        if lbl is None or not lbl.exists():
            alt = img.with_suffix(".txt")
            lbl = alt if alt.exists() else lbl
        if lbl is None or not lbl.exists():
            if len(issues["images_without_labels"]) < 25:
                issues["images_without_labels"].append(str(img))
            continue
        issues["n_labels"] += 1
        try:
            lines = [l for l in lbl.read_text().splitlines() if l.strip()]
        except Exception:
            continue
        if not lines:
            issues["empty_labels"] += 1
            continue
        for line in lines:
            f = line.split()
            if len(f) < 5:
                if len(issues["invalid_boxes"]) < 25:
                    issues["invalid_boxes"].append(f"{lbl}: {line}")
                continue
            cid = f[0]
            issues["class_ids"][cid] = issues["class_ids"].get(cid, 0) + 1
            try:
                vals = [float(x) for x in f[1:5]]
            except ValueError:
                if len(issues["invalid_boxes"]) < 25:
                    issues["invalid_boxes"].append(f"{lbl}: {line}")
                continue
            if any(v < 0.0 or v > 1.0 for v in vals) or vals[2] <= 0 or vals[3] <= 0:
                if len(issues["invalid_boxes"]) < 25:
                    issues["invalid_boxes"].append(f"{lbl}: {line}")
    return issues


def duplicate_filenames(root: Path, exts):
    seen, dups = {}, []
    if not root.exists():
        return dups
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in exts:
            seen.setdefault(p.name, []).append(p)
    for name, paths in seen.items():
        if len(paths) > 1:
            dups.append((name, len(paths)))
    return sorted(dups, key=lambda x: -x[1])


def empty_dirs(root: Path):
    out = []
    if not root.exists():
        return out
    for p in root.rglob("*"):
        if p.is_dir() and not any(p.iterdir()):
            out.append(str(p))
    return out


print("=" * 60)
print("DATASET INTEGRITY")
print("=" * 60)

for label, root in [("VIOLENCE", VIOLENCE_DATA)]:
    if not root.exists():
        print(f"\n{label}: directory missing - skipped")
        continue
    v, inv, bad = verify_images(root)
    dups = duplicate_filenames(root, IMAGE_EXTS)
    ed = empty_dirs(root)
    print(f"\n{label}")
    print(f"  valid images       : {v}")
    print(f"  CORRUPT images     : {inv}")
    print(f"  duplicate filenames: {len(dups)}")
    print(f"  empty directories  : {len(ed)}")
    for b in bad[:5]:
        print("     corrupt ->", b)
    INTEGRITY[label] = {"valid": v, "invalid": inv, "duplicates": len(dups),
                        "empty_dirs": len(ed), "bad_sample": bad[:25]}

if AUDIO_DATA.exists():
    v, inv, bad, durs = verify_audio(AUDIO_DATA)
    print("\nAUDIO")
    print(f"  readable clips     : {v}")
    print(f"  UNREADABLE clips   : {inv}")
    if durs:
        durs_sorted = sorted(durs)
        print(f"  duration min/med/max: {durs_sorted[0]:.2f}s / "
              f"{durs_sorted[len(durs_sorted)//2]:.2f}s / {durs_sorted[-1]:.2f}s")
        print(f"  clips shorter than 1s: {sum(1 for d in durs if d < 1.0)}")
    for b in bad[:5]:
        print("     unreadable ->", b)
    INTEGRITY["AUDIO"] = {"valid": v, "invalid": inv, "bad_sample": bad[:25]}

for label, root in [("WEAPON", WEAPON_DATA), ("FIRE_SMOKE", FIRE_SMOKE_DATA)]:
    if not root.exists():
        print(f"\n{label}: directory missing - skipped")
        continue
    iss = check_yolo_labels(root)
    print(f"\n{label} (YOLO)")
    print(f"  images                 : {iss['n_images']}")
    print(f"  label files matched    : {iss['n_labels']}")
    print(f"  images WITHOUT labels  : {len(iss['images_without_labels'])}"
          f"{' (first 25 listed)' if iss['images_without_labels'] else ''}")
    print(f"  empty label files      : {iss['empty_labels']}")
    print(f"  INVALID boxes          : {len(iss['invalid_boxes'])}")
    print(f"  class id histogram     : {iss['class_ids']}")
    INTEGRITY[label] = {k: (v if not isinstance(v, list) else v[:25]) for k, v in iss.items()}

# ---- train / validation leakage (same file content appearing in both splits) ----
def leakage_check(root: Path, exts, train_keys=("train",), val_keys=("val", "valid")):
    def digest(p):
        h = hashlib.md5()
        with open(p, "rb") as f:
            h.update(f.read(200_000))
        return h.hexdigest()

    def collect(keys):
        out = {}
        for p in root.rglob("*"):
            if p.is_file() and p.suffix.lower() in exts:
                low = [s.lower() for s in p.parts]
                if any(k in low for k in keys):
                    try:
                        out[digest(p)] = p
                    except Exception:
                        pass
        return out

    if not root.exists():
        return None
    tr, va = collect(train_keys), collect(val_keys)
    if not tr or not va:
        return None
    overlap = set(tr) & set(va)
    return {"train": len(tr), "val": len(va), "overlap": len(overlap),
            "examples": [str(tr[h]) for h in list(overlap)[:5]]}


print("\nTRAIN / VAL LEAKAGE (content hash)")
for label, root, exts in [("VIOLENCE", VIOLENCE_DATA, IMAGE_EXTS),
                          ("AUDIO", AUDIO_DATA, AUDIO_EXTS),
                          ("WEAPON", WEAPON_DATA, IMAGE_EXTS),
                          ("FIRE_SMOKE", FIRE_SMOKE_DATA, IMAGE_EXTS)]:
    res = leakage_check(root, exts)
    if res is None:
        print(f"  {label:11}: no explicit train/val folders - split is done in-notebook (seeded, disjoint)")
    else:
        flag = "  <-- LEAKAGE" if res["overlap"] else "  OK"
        print(f"  {label:11}: train={res['train']} val={res['val']} overlap={res['overlap']}{flag}")
        for ex in res["examples"]:
            print("        ", ex)
        INTEGRITY.setdefault(label, {})["leakage"] = res

(V2_RUNS_DIR / f"integrity_{RUN_ID}.json").write_text(
    json.dumps(INTEGRITY, indent=2, default=str), encoding="utf-8")
print("\nIntegrity report saved to:", V2_RUNS_DIR / f"integrity_{RUN_ID}.json")

DATASET INTEGRITY

VIOLENCE
  valid images       : 19610
  CORRUPT images     : 19610
  duplicate filenames: 0
  empty directories  : 0
     corrupt -> G:\Capstone\data\violence_data\fight\._rwf_train_fight_-1l5631l3fg_0_00.jpg
     corrupt -> G:\Capstone\data\violence_data\fight\._rwf_train_fight_-1l5631l3fg_0_01.jpg
     corrupt -> G:\Capstone\data\violence_data\fight\._rwf_train_fight_-1l5631l3fg_0_02.jpg
     corrupt -> G:\Capstone\data\violence_data\fight\._rwf_train_fight_-1l5631l3fg_0_03.jpg
     corrupt -> G:\Capstone\data\violence_data\fight\._rwf_train_fight_-1l5631l3fg_0_04.jpg

AUDIO
  readable clips     : 467
  UNREADABLE clips   : 9
  duration min/med/max: 0.00s / 10.00s / 10.00s
  clips shorter than 1s: 92
     unreadable -> G:\Capstone\data\audio_data\audioset_clips\scream\aslxfM0T8No_10.0.wav
     unreadable -> G:\Capstone\data\audio_data\audioset_clips\scream\aVdPh-RiNcc_0.0.temp.m4a
     unreadable -> G:\Capstone\data\audio_data\audioset_clips\siren\4lflH2Jvm8k_30.0.

---
# CELL 7 — GLOBAL TRAINING POLICY

## REAL DATA POLICY

This notebook strictly prohibits:

1. Synthetic datasets
2. Synthetic images
3. Synthetic audio
4. Synthetic spectrograms
5. Randomly generated multimodal samples
6. Artificial TCI targets
7. Artificial ground-truth labels

Allowed:

1. Real dataset samples
2. Real dataset augmentation
3. Resizing
4. Cropping
5. Horizontal flipping
6. Brightness/contrast augmentation
7. Audio time shifting
8. Audio amplitude augmentation
9. Audio noise augmentation using real/background recordings
10. Normal preprocessing

**Every training sample must trace back to a real source file.**

Each dataset built in this notebook records the absolute source path of every sample in a
`sample_manifest.csv` inside its run directory, so any prediction can be traced back to a file on disk.

---
# CELL 7b — Real-data guards

These helpers are the mechanical enforcement of the policy above.

In [6]:
BANNED_SOURCE_MARKERS = [
    "SyntheticAudioDataset", "generate_synthetic", "synthetic_scenario",
    "make_synthetic", "SyntheticDataset", "fake_dataset", "random_tci",
]

class SyntheticDataError(RuntimeError):
    pass


def assert_real_file(path):
    # Every sample must be an existing file on disk. No exceptions.
    p = Path(path)
    if not p.exists() or not p.is_file():
        raise SyntheticDataError(f"Sample does not trace back to a real file on disk: {p}")
    return p


def write_sample_manifest(run_dir: Path, split: str, items):
    # items: list of (abs_path, label)
    out = run_dir / f"sample_manifest_{split}.csv"
    with open(out, "w", encoding="utf-8", newline="") as f:
        f.write("source_path,label\n")
        for p, y in items:
            f.write(f"{str(p).replace(',', '%2C')},{y}\n")
    return out


def set_seed(seed=SEED):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import numpy as _np
        _np.random.seed(seed)
    except Exception:
        pass
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print("Real-data guards active. Seed set to", SEED)

Real-data guards active. Seed set to 42


---
# CELL 7c — Shared training helpers

Dataset construction from **real files only**, a common image-classifier training loop with
class weighting / early stopping / checkpointing, and a standard metrics report used by every module.

In [7]:
import numpy as np
import pandas as pd
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report,
                             average_precision_score, roc_auc_score,
                             precision_recall_curve, mean_absolute_error,
                             mean_squared_error, r2_score)

PATH_COLS  = ["frame_path", "image_path", "filepath", "file_path", "path",
              "file", "filename", "image", "img", "clip_path", "audio_path", "wav_path"]
LABEL_COLS = ["label", "class", "target", "category", "y", "class_name", "labels"]


def _pick_col(df, candidates):
    lower = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        if cand in lower:
            return lower[cand]
    for c in df.columns:
        if any(cand in c.lower() for cand in candidates):
            return c
    return None


def _resolve(root: Path, raw: str):
    # Resolve a manifest path entry against the dataset root. Real files only.
    s = str(raw).strip().strip('"').replace("\\", "/")
    cands = [Path(s), root / s, root / Path(s).name]
    # also try root/<class>/<name> style and a two-level tail
    tail = Path(s)
    if len(tail.parts) >= 2:
        cands.append(root / Path(*tail.parts[-2:]))
    if len(tail.parts) >= 3:
        cands.append(root / Path(*tail.parts[-3:]))
    for c in cands:
        try:
            if c.exists() and c.is_file():
                return c
        except OSError:
            continue
    return None


def load_items_from_csv(csv_path: Path, root: Path, classes, exts):
    # Returns (items, stats). items = [(abs_path, class_index)]
    df = pd.read_csv(csv_path)
    pcol = _pick_col(df, PATH_COLS)
    lcol = _pick_col(df, LABEL_COLS)
    if pcol is None:
        raise RuntimeError(f"{csv_path.name}: no path-like column found in {list(df.columns)}")
    cls_lower = {c.lower(): i for i, c in enumerate(classes)}
    items, unresolved, unmapped = [], 0, 0
    for _, row in df.iterrows():
        p = _resolve(root, row[pcol])
        if p is None:
            unresolved += 1
            continue
        y = None
        if lcol is not None and not pd.isna(row[lcol]):
            raw = str(row[lcol]).strip().lower()
            if raw in cls_lower:
                y = cls_lower[raw]
            else:
                try:
                    iv = int(float(raw))
                    if 0 <= iv < len(classes):
                        y = iv
                except ValueError:
                    y = None
        if y is None:  # fall back to folder name
            for part in reversed(p.parts):
                if part.lower() in cls_lower:
                    y = cls_lower[part.lower()]
                    break
        if y is None:
            unmapped += 1
            continue
        items.append((p, y))
    return items, {"rows": len(df), "resolved": len(items),
                   "unresolved_paths": unresolved, "unmapped_labels": unmapped,
                   "path_column": pcol, "label_column": lcol}


def load_items_from_dirs(root: Path, classes, exts):
    dirs = find_class_dirs(root, classes)
    items = []
    for i, c in enumerate(classes):
        for d in dirs[c]:
            for p in d.rglob("*"):
                if p.is_file() and p.suffix.lower() in exts:
                    items.append((p, i))
    # de-duplicate identical absolute paths
    seen, uniq = set(), []
    for p, y in items:
        rp = str(p.resolve()).lower()
        if rp not in seen:
            seen.add(rp)
            uniq.append((p, y))
    return uniq


def build_splits(root: Path, classes, exts, train_csv: Path, val_csv: Path,
                 val_fraction=0.2, run_dir: Path = None, name="dataset"):
    # Prefer real manifests; otherwise scan real directories; never generate.
    info = {"source": None}
    train_items, val_items = [], []

    if train_csv is not None and train_csv.exists():
        train_items, st = load_items_from_csv(train_csv, root, classes, exts)
        info["train_csv"] = st
        info["source"] = "csv"
        if val_csv is not None and val_csv.exists():
            val_items, sv = load_items_from_csv(val_csv, root, classes, exts)
            info["val_csv"] = sv

    if not train_items:
        all_items = load_items_from_dirs(root, classes, exts)
        info["source"] = "directory_scan"
        info["scanned"] = len(all_items)
        if not all_items:
            raise RuntimeError(
                f"No real samples found for '{name}' under {root}. "
                "This notebook will not synthesise data - fix the dataset path/layout."
            )
        # keep any pre-existing val/ split honest if one exists on disk
        def in_split(p, keys):
            low = [s.lower() for s in Path(p).parts]
            return any(k in low for k in keys)
        pre_train = [it for it in all_items if in_split(it[0], ("train",))]
        pre_val   = [it for it in all_items if in_split(it[0], ("val", "valid"))]
        if pre_train and pre_val:
            train_items, val_items = pre_train, pre_val
            info["split"] = "existing on-disk train/val folders"
        else:
            ys = [y for _, y in all_items]
            tr, va = train_test_split(all_items, test_size=val_fraction,
                                      random_state=SEED, stratify=ys)
            train_items, val_items = tr, va
            info["split"] = f"stratified in-notebook split (seed={SEED}, val={val_fraction})"

    if not val_items:
        ys = [y for _, y in train_items]
        train_items, val_items = train_test_split(
            train_items, test_size=val_fraction, random_state=SEED, stratify=ys)
        info["split"] = f"stratified in-notebook split (seed={SEED}, val={val_fraction})"

    # hard guarantee: disjoint splits
    tr_set = {str(p.resolve()).lower() for p, _ in train_items}
    val_items = [(p, y) for p, y in val_items if str(p.resolve()).lower() not in tr_set]

    # every sample must be a real file
    for p, _ in train_items[:2000] + val_items[:2000]:
        assert_real_file(p)

    if run_dir is not None:
        write_sample_manifest(run_dir, "train", train_items)
        write_sample_manifest(run_dir, "val", val_items)

    def dist(items):
        d = {c: 0 for c in classes}
        for _, y in items:
            d[classes[y]] += 1
        return d

    info["train_size"] = len(train_items)
    info["val_size"] = len(val_items)
    info["train_distribution"] = dist(train_items)
    info["val_distribution"] = dist(val_items)
    return train_items, val_items, info


class RealImageDataset(Dataset):
    # Loads ONLY real image files listed in `items`.
    def __init__(self, items, transform):
        self.items = items
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, label = self.items[idx]
        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            # A corrupt real file is skipped by substituting another REAL file,
            # never by generating an image.
            alt = self.items[(idx + 1) % len(self.items)]
            img = Image.open(alt[0]).convert("RGB")
            label = alt[1]
        return self.transform(img), label


def class_weights_from_items(items, num_classes):
    counts = np.bincount([y for _, y in items], minlength=num_classes).astype(np.float64)
    counts[counts == 0] = 1.0
    w = counts.sum() / (num_classes * counts)
    return torch.tensor(w, dtype=torch.float32)


def classification_report_dict(y_true, y_pred, y_prob, classes):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred); y_prob = np.asarray(y_prob)
    out = {}
    out["accuracy"] = float(accuracy_score(y_true, y_pred))
    p, r, f1, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(len(classes))), zero_division=0)
    out["per_class"] = {
        classes[i]: {"precision": float(p[i]), "recall": float(r[i]),
                     "f1": float(f1[i]), "support": int(sup[i])}
        for i in range(len(classes))
    }
    mp, mr, mf1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    out["macro_precision"], out["macro_recall"], out["macro_f1"] = float(mp), float(mr), float(mf1)
    wp, wr, wf1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0)
    out["weighted_f1"] = float(wf1)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(classes))))
    out["confusion_matrix"] = cm.tolist()
    out["classes"] = list(classes)
    if len(classes) == 2:
        pos = y_prob[:, 1]
        out["pr_auc"] = float(average_precision_score(y_true, pos))
        try:
            out["roc_auc"] = float(roc_auc_score(y_true, pos))
        except Exception:
            out["roc_auc"] = None
        tn, fp, fn, tp = cm.ravel()
        out["false_negative_rate"] = float(fn / (fn + tp)) if (fn + tp) else 0.0
        out["false_positive_rate"] = float(fp / (fp + tn)) if (fp + tn) else 0.0
        # Operating threshold that maximises F1 on the validation set
        prec, rec, thr = precision_recall_curve(y_true, pos)
        f1s = np.divide(2 * prec * rec, prec + rec,
                        out=np.zeros_like(prec), where=(prec + rec) > 0)
        best = int(np.nanargmax(f1s[:-1])) if len(thr) else 0
        out["best_threshold"] = float(thr[best]) if len(thr) else 0.5
        out["best_threshold_f1"] = float(f1s[best]) if len(thr) else out["macro_f1"]
        # recall-first threshold: lowest threshold giving <= 5% FNR
        target = 0.95
        idx = np.where(rec[:-1] >= target)[0]
        out["threshold_at_95_recall"] = float(thr[idx[-1]]) if len(idx) and len(thr) else None
    else:
        out["pr_auc"] = None
    out["report_text"] = classification_report(
        y_true, y_pred, labels=list(range(len(classes))),
        target_names=classes, zero_division=0)
    return out


def print_metrics(title, m):
    print("\n" + "=" * 60)
    print(f"{title} — VALIDATION METRICS")
    print("=" * 60)
    print(f"Accuracy    : {m['accuracy']:.4f}")
    print(f"Macro F1    : {m['macro_f1']:.4f}   (P {m['macro_precision']:.4f} / R {m['macro_recall']:.4f})")
    if m.get("pr_auc") is not None:
        print(f"PR-AUC      : {m['pr_auc']:.4f}")
    if "false_negative_rate" in m:
        print(f"FNR         : {m['false_negative_rate']:.4f}")
        print(f"Best thresh : {m['best_threshold']:.3f} (F1 {m['best_threshold_f1']:.4f})")
        if m.get("threshold_at_95_recall") is not None:
            print(f"Thresh @95% recall: {m['threshold_at_95_recall']:.3f}")
    print("\nConfusion matrix (rows = true, cols = pred):")
    hdr = " " * 14 + "".join(f"{c:>12}" for c in m["classes"])
    print(hdr)
    for i, row in enumerate(m["confusion_matrix"]):
        print(f"{m['classes'][i]:>13} " + "".join(f"{v:>12}" for v in row))
    print("\n" + m["report_text"])


from contextlib import nullcontext

def amp_ctx():
    if USE_AMP:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def make_scaler():
    # torch >= 2.4 moved GradScaler to torch.amp; keep both paths working.
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)


@torch.no_grad()
def evaluate_classifier(model, loader, num_classes, device=None):
    device = device or DEVICE
    model.eval()
    ys, probs = [], []
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        with amp_ctx():
            out = model(xb)
        probs.append(torch.softmax(out.float(), dim=1).cpu().numpy())
        ys.append(yb.numpy())
    y_true = np.concatenate(ys)
    y_prob = np.concatenate(probs)
    return y_true, y_prob, y_prob.argmax(1)


def save_metadata(model_path: Path, meta: dict):
    out = model_path.with_name(model_path.stem + "_metadata.json")
    meta = dict(meta)
    meta.setdefault("created", datetime.now().isoformat(timespec="seconds"))
    meta.setdefault("run_id", RUN_ID)
    meta.setdefault("training_type", "real_data")
    meta["synthetic_data"] = False
    out.write_text(json.dumps(meta, indent=4, default=str), encoding="utf-8")
    print("Metadata written:", out)
    return out


print("Shared helpers loaded.")

Shared helpers loaded.


---
# CELL 7d — Generic image-classifier trainer

Used by Module 1 (violence). Improvements over the previous pipeline:
class weighting, cosine LR schedule, AMP, per-epoch validation on **macro F1** (not raw accuracy),
best-checkpoint saving, early stopping, and a full metrics report with threshold selection.

In [8]:
def make_image_transforms(img_size):
    train_transform = transforms.Compose([
        transforms.Resize((int(img_size * 1.15), int(img_size * 1.15))),
        transforms.RandomResizedCrop(img_size, scale=(0.80, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.20, contrast=0.20, saturation=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.15, scale=(0.02, 0.10)),
    ])
    eval_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])
    return train_transform, eval_transform


def build_resnet18(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    in_f = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.30),
        nn.Linear(in_f, num_classes),
    )
    return model


def train_image_classifier(name, train_items, val_items, classes, img_size,
                           batch_size, epochs, lr, weight_decay, patience,
                           run_dir: Path, out_model: Path, dataset_dir: Path,
                           split_info: dict):
    set_seed(SEED)
    num_classes = len(classes)
    train_tf, eval_tf = make_image_transforms(img_size)

    train_ds = RealImageDataset(train_items, train_tf)
    val_ds   = RealImageDataset(val_items, eval_tf)
    train_ld = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"),
                          drop_last=False)
    val_ld   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))

    model = build_resnet18(num_classes).to(DEVICE)
    weights = class_weights_from_items(train_items, num_classes).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = make_scaler()

    print("=" * 60)
    print(f"TRAINING: {name}")
    print("=" * 60)
    print("Dataset      :", dataset_dir)
    print("Classes      :", classes)
    print("Train / Val  :", len(train_items), "/", len(val_items))
    print("Class weights:", [round(float(w), 4) for w in weights.cpu()])
    print("Device       :", DEVICE, "| AMP:", USE_AMP)
    print()

    history, best_f1, best_epoch, bad_epochs = [], -1.0, -1, 0
    ckpt_path = run_dir / "best_checkpoint.pt"
    t_start = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        running, seen, correct = 0.0, 0, 0
        t0 = time.time()
        for xb, yb in train_ld:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with amp_ctx():
                out = model(xb)
                loss = criterion(out, yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            running += loss.item() * xb.size(0)
            seen += xb.size(0)
            correct += (out.float().argmax(1) == yb).sum().item()
        scheduler.step()

        y_true, y_prob, y_pred = evaluate_classifier(model, val_ld, num_classes)
        m = classification_report_dict(y_true, y_pred, y_prob, classes)
        row = {"epoch": epoch, "train_loss": running / max(seen, 1),
               "train_acc": correct / max(seen, 1),
               "val_acc": m["accuracy"], "val_macro_f1": m["macro_f1"],
               "lr": optimizer.param_groups[0]["lr"], "seconds": time.time() - t0}
        history.append(row)
        flag = ""
        if m["macro_f1"] > best_f1 + 1e-5:
            best_f1, best_epoch, bad_epochs = m["macro_f1"], epoch, 0
            torch.save({
                "model_state_dict": model.state_dict(),
                "classes": classes,
                "architecture": "resnet18",
                "image_size": img_size,
                "normalization": {"mean": [0.485, 0.456, 0.406],
                                  "std": [0.229, 0.224, 0.225]},
                "epoch": epoch,
                "val_macro_f1": best_f1,
            }, ckpt_path)
            flag = "  <-- best (checkpoint saved)"
        else:
            bad_epochs += 1
        print(f"epoch {epoch:>3}/{epochs} | loss {row['train_loss']:.4f} | "
              f"train_acc {row['train_acc']:.4f} | val_acc {row['val_acc']:.4f} | "
              f"val_macroF1 {row['val_macro_f1']:.4f} | {row['seconds']:.1f}s{flag}")
        if bad_epochs >= patience:
            print(f"Early stopping at epoch {epoch} (no macro-F1 improvement for {patience} epochs).")
            break

    total_time = time.time() - t_start

    # Reload the best checkpoint and produce the final report
    ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck["model_state_dict"])
    y_true, y_prob, y_pred = evaluate_classifier(model, val_ld, num_classes)
    final = classification_report_dict(y_true, y_pred, y_prob, classes)
    final["best_epoch"] = best_epoch
    final["epochs_run"] = len(history)
    final["training_seconds"] = round(total_time, 1)
    print_metrics(name, final)

    # Save the production checkpoint under the NEW V2 name only
    torch.save({
        "model_state_dict": model.state_dict(),
        "classes": classes,
        "architecture": "resnet18",
        "image_size": img_size,
        "normalization": {"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]},
        "input_format": "RGB, float32, NCHW, ImageNet-normalised",
        "best_threshold": final.get("best_threshold"),
        "val_macro_f1": final["macro_f1"],
        "run_id": RUN_ID,
        "training_type": "real_data",
        "synthetic_data": False,
    }, out_model)
    print("\nModel saved to:", out_model)

    pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
    (run_dir / "metrics.json").write_text(
        json.dumps(final, indent=2, default=str), encoding="utf-8")
    (run_dir / "split_info.json").write_text(
        json.dumps(split_info, indent=2, default=str), encoding="utf-8")

    meta = {
        "model_name": out_model.stem,
        "architecture": "ResNet18",
        "dataset": str(dataset_dir),
        "classes": list(classes),
        "image_size": img_size,
        "normalization": {"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]},
        "color_space": "RGB",
        "batch_size": batch_size,
        "epochs_configured": epochs,
        "epochs_run": len(history),
        "best_epoch": best_epoch,
        "learning_rate": lr,
        "weight_decay": weight_decay,
        "class_weighting": True,
        "label_smoothing": 0.05,
        "seed": SEED,
        "device": DEVICE,
        "train_samples": len(train_items),
        "val_samples": len(val_items),
        "split_info": split_info,
        "metrics": {k: final[k] for k in
                    ["accuracy", "macro_f1", "macro_precision", "macro_recall",
                     "weighted_f1", "pr_auc", "false_negative_rate",
                     "best_threshold", "confusion_matrix"] if k in final},
        "training_type": "real_data",
        "synthetic_data": False,
    }
    meta_path = save_metadata(out_model, meta)
    return model, final, meta_path


print("Image trainer ready.")

Image trainer ready.


---
---
# MODULE 1 — VIOLENCE

`fight` → 1  ·  `nofight` → 0

## CELL 8 — Violence configuration

In [9]:
VIOLENCE_TRAIN_CSV     = VIOLENCE_DATA / "train.csv"
VIOLENCE_VAL_CSV       = VIOLENCE_DATA / "val.csv"
VIOLENCE_CLASSES       = ["nofight", "fight"]        # index 0 = nofight, 1 = fight
VIOLENCE_IMAGE_SIZE    = 224
VIOLENCE_BATCH_SIZE    = 32
VIOLENCE_EPOCHS        = 15
VIOLENCE_LR            = 1e-4
VIOLENCE_WEIGHT_DECAY  = 1e-3
VIOLENCE_PATIENCE      = 5

VIOLENCE_RUN_DIR = V2_RUNS_DIR / "violence"
VIOLENCE_RUN_DIR.mkdir(parents=True, exist_ok=True)

print("VIOLENCE dataset :", VIOLENCE_DATA)
print("train.csv        :", VIOLENCE_TRAIN_CSV, "| exists:", VIOLENCE_TRAIN_CSV.exists())
print("val.csv          :", VIOLENCE_VAL_CSV, "| exists:", VIOLENCE_VAL_CSV.exists())
print("classes          :", VIOLENCE_CLASSES)
print("run dir          :", VIOLENCE_RUN_DIR)
print("output model     :", VIOLENCE_MODEL_V3)

VIOLENCE dataset : G:\Capstone\data\violence_data
train.csv        : G:\Capstone\data\violence_data\train.csv | exists: True
val.csv          : G:\Capstone\data\violence_data\val.csv | exists: True
classes          : ['nofight', 'fight']
run dir          : G:\Sentrix\training\runs_v3_real\violence
output model     : G:\Sentrix\backend\models\v3_real\violence_classifier_real_v3.pt


## CELL 9 — Load REAL violence data

Reads `train.csv` / `val.csv` and resolves every `frame_path` against `G:\Capstone\data\violence_data`.
If the manifests are absent it scans the real `fight/` and `nofight/` directories instead.

**Frames are never generated.** Every row that cannot be resolved to a file on disk is reported and dropped.

In [10]:
violence_train, violence_val, violence_split_info = build_splits(
    root=VIOLENCE_DATA,
    classes=VIOLENCE_CLASSES,
    exts=IMAGE_EXTS,
    train_csv=VIOLENCE_TRAIN_CSV,
    val_csv=VIOLENCE_VAL_CSV,
    val_fraction=0.2,
    run_dir=VIOLENCE_RUN_DIR,
    name="violence",
)

print("=" * 60)
print("VIOLENCE — REAL DATA SPLIT")
print("=" * 60)
print(json.dumps(violence_split_info, indent=2, default=str))
print("\nSample of real source files used for training:")
for p, y in violence_train[:5]:
    print(f"  [{VIOLENCE_CLASSES[y]:8}] {p}")

VIOLENCE — REAL DATA SPLIT
{
  "source": "csv",
  "train_csv": {
    "rows": 15680,
    "resolved": 15680,
    "unresolved_paths": 0,
    "unmapped_labels": 0,
    "path_column": "frame_path",
    "label_column": "label"
  },
  "val_csv": {
    "rows": 3930,
    "resolved": 3930,
    "unresolved_paths": 0,
    "unmapped_labels": 0,
    "path_column": "frame_path",
    "label_column": "label"
  },
  "train_size": 15680,
  "val_size": 3930,
  "train_distribution": {
    "nofight": 7600,
    "fight": 8080
  },
  "val_distribution": {
    "nofight": 1900,
    "fight": 2030
  }
}

Sample of real source files used for training:
  [fight   ] G:\Capstone\data\violence_data\fight\rwf_train_fight_aIg9Yb0O7Hg_5_00.jpg
  [fight   ] G:\Capstone\data\violence_data\fight\rwf_train_fight_videoplayback (2)_66_06.jpg
  [nofight ] G:\Capstone\data\violence_data\nofight\rwf_train_nofight_UIsigi4O-sI_1_00.jpg
  [fight   ] G:\Capstone\data\violence_data\fight\rwf_train_fight_umOs1WLBkuo_1_06.jpg
  [nofight 

## CELL 10 — Violence augmentation (real images only)

Augmentation of **real images** — resize, random crop, horizontal flip, colour jitter,
random erasing. No synthetic frame generation.

In [11]:
train_transform, eval_transform = make_image_transforms(VIOLENCE_IMAGE_SIZE)
print("TRAIN transform:")
print(train_transform)
print("\nEVAL / INFERENCE transform (must match the SENTRIX application):")
print(eval_transform)

TRAIN transform:
Compose(
    Resize(size=(257, 257), interpolation=bilinear, max_size=None, antialias=True)
    RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.8, 1.2), contrast=(0.8, 1.2), saturation=(0.85, 1.15), hue=None)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    RandomErasing(p=0.15, scale=(0.02, 0.1), ratio=(0.3, 3.3), value=0, inplace=False)
)

EVAL / INFERENCE transform (must match the SENTRIX application):
Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


## CELL 11 — Train ResNet18 (violence)

```text
G:\Capstone\data\violence_data
        ↓
   real frames
        ↓
    ResNet18
        ↓
 fight probability
        ↓
G:\Sentrix\backend\models\v3_real\violence_classifier_real_v3.pt
```

In [12]:
try:
    violence_model, violence_metrics, violence_meta_path = train_image_classifier(
        name="VIOLENCE",
        train_items=violence_train,
        val_items=violence_val,
        classes=VIOLENCE_CLASSES,
        img_size=VIOLENCE_IMAGE_SIZE,
        batch_size=VIOLENCE_BATCH_SIZE,
        epochs=VIOLENCE_EPOCHS,
        lr=VIOLENCE_LR,
        weight_decay=VIOLENCE_WEIGHT_DECAY,
        patience=VIOLENCE_PATIENCE,
        run_dir=VIOLENCE_RUN_DIR,
        out_model=VIOLENCE_MODEL_V3,
        dataset_dir=VIOLENCE_DATA,
        split_info=violence_split_info,
    )
    RESULTS["violence"] = violence_metrics
    ARTIFACTS["violence"] = {"model": VIOLENCE_MODEL_V3, "metadata": violence_meta_path,
                             "run_dir": VIOLENCE_RUN_DIR}
    MODULE_STATUS["violence"] = "TRAINED"
except Exception as e:
    MODULE_STATUS["violence"] = "FAILED"
    print("VIOLENCE TRAINING FAILED:", type(e).__name__, e)
    raise

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\PC/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:12<00:00, 3.81MB/s]


TRAINING: VIOLENCE
Dataset      : G:\Capstone\data\violence_data
Classes      : ['nofight', 'fight']
Train / Val  : 15680 / 3930
Class weights: [1.0316, 0.9703]
Device       : cuda | AMP: True

epoch   1/15 | loss 0.3991 | train_acc 0.8425 | val_acc 0.7817 | val_macroF1 0.7805 | 319.5s  <-- best (checkpoint saved)
epoch   2/15 | loss 0.2554 | train_acc 0.9367 | val_acc 0.8046 | val_macroF1 0.8040 | 144.5s  <-- best (checkpoint saved)
epoch   3/15 | loss 0.2115 | train_acc 0.9614 | val_acc 0.8104 | val_macroF1 0.8099 | 271.1s  <-- best (checkpoint saved)
epoch   4/15 | loss 0.1863 | train_acc 0.9752 | val_acc 0.8293 | val_macroF1 0.8283 | 431.1s  <-- best (checkpoint saved)
epoch   5/15 | loss 0.1754 | train_acc 0.9798 | val_acc 0.8056 | val_macroF1 0.8047 | 178.6s
epoch   6/15 | loss 0.1696 | train_acc 0.9821 | val_acc 0.8244 | val_macroF1 0.8240 | 135.1s
epoch   7/15 | loss 0.1593 | train_acc 0.9883 | val_acc 0.8305 | val_macroF1 0.8300 | 134.3s  <-- best (checkpoint saved)
epoch   8/

---
---
# MODULE 2 — AUDIO

> The previous pipeline trained this model on **mathematically generated spectrograms**
> (`SyntheticAudioDataset`). None of that exists here. Every spectrogram below is computed
> from a real `.wav` file on disk.

## CELL 17 — Audio configuration

In [15]:
AUDIO_CLASSES = [
    "speech_normal",   # 0
    "siren",           # 1
    "fire",            # 2
    "scream"           # 3
]

AUDIO_SAMPLE_RATE   = 16000
AUDIO_DURATION      = 4.0
AUDIO_N_SAMPLES     = int(AUDIO_SAMPLE_RATE * AUDIO_DURATION)
AUDIO_N_MELS        = 64
AUDIO_N_FFT         = 1024
AUDIO_HOP_LENGTH    = 256
AUDIO_F_MIN         = 20
AUDIO_F_MAX         = AUDIO_SAMPLE_RATE // 2
AUDIO_TOP_DB        = 80

AUDIO_BATCH_SIZE    = 32
AUDIO_EPOCHS        = 25
AUDIO_LR            = 1e-3
AUDIO_WEIGHT_DECAY  = 1e-4
AUDIO_PATIENCE      = 7
AUDIO_VAL_FRACTION  = 0.2

AUDIO_TIME_FRAMES   = AUDIO_N_SAMPLES // AUDIO_HOP_LENGTH + 1

AUDIO_RUN_DIR = V2_RUNS_DIR / "audio"
AUDIO_RUN_DIR.mkdir(parents=True, exist_ok=True)

# the manifest may sit beside the clips or one level up
_audio_manifest_candidates = [
    AUDIO_DATA / "download_manifest.csv",
    AUDIO_DATA.parent / "download_manifest.csv",
]
AUDIO_MANIFEST = next((p for p in _audio_manifest_candidates if p.exists()),
                      _audio_manifest_candidates[0])

# Explicitly NOT used unless you deliberately decide to incorporate AudioSet later:
AUDIOSET_IGNORED = ["balanced_train_segments.csv", "eval_segments.csv",
                    "unbalanced_train_segments.csv"]

print("AUDIO classes   :", AUDIO_CLASSES)
print("manifest        :", AUDIO_MANIFEST, "| exists:", AUDIO_MANIFEST.exists())
print("sample rate     :", AUDIO_SAMPLE_RATE)
print("clip length     :", AUDIO_DURATION, "s =", AUDIO_N_SAMPLES, "samples")
print("mel spectrogram : n_mels=%d n_fft=%d hop=%d -> input shape (1, %d, %d)"
      % (AUDIO_N_MELS, AUDIO_N_FFT, AUDIO_HOP_LENGTH, AUDIO_N_MELS, AUDIO_TIME_FRAMES))
print("run dir         :", AUDIO_RUN_DIR)
print("output model    :", AUDIO_MODEL_V3)
print("\nAudioSet manifests deliberately IGNORED:", AUDIOSET_IGNORED)

AUDIO classes   : ['speech_normal', 'siren', 'fire', 'scream']
manifest        : G:\Capstone\data\audio_data\download_manifest.csv | exists: True
sample rate     : 16000
clip length     : 4.0 s = 64000 samples
mel spectrogram : n_mels=64 n_fft=1024 hop=256 -> input shape (1, 64, 251)
run dir         : G:\Sentrix\training\runs_v3_real\audio
output model    : G:\Sentrix\backend\models\v3_real\audio_classifier_real_v3.pt

AudioSet manifests deliberately IGNORED: ['balanced_train_segments.csv', 'eval_segments.csv', 'unbalanced_train_segments.csv']


## CELL 18 — Audio manifest (real WAV files only)

Uses `G:\Capstone\data\audio_data\download_manifest.csv` plus the actual `.wav` files.
`balanced_train_segments.csv` / `eval_segments.csv` are **not** used.

Class mapping:

```text
speech_normal → 0
siren         → 1
explosion     → 2
fire          → 3
scream        → 4
```

In [16]:
audio_items, audio_source_info = [], {}

if AUDIO_MANIFEST.exists():
    try:
        audio_items, audio_source_info = load_items_from_csv(
            AUDIO_MANIFEST, AUDIO_DATA, AUDIO_CLASSES, AUDIO_EXTS)
        audio_source_info["source"] = f"manifest: {AUDIO_MANIFEST.name}"
    except Exception as e:
        print("Manifest could not be parsed:", e)

if not audio_items:
    audio_items = load_items_from_dirs(AUDIO_DATA, AUDIO_CLASSES, AUDIO_EXTS)
    audio_source_info = {"source": "directory_scan", "resolved": len(audio_items)}

if not audio_items:
    raise RuntimeError(
        "No real audio files found under " + str(AUDIO_DATA) +
        ". This notebook will NOT generate synthetic audio or spectrograms."
    )

for p, _ in audio_items[:500]:
    assert_real_file(p)

counts = {c: 0 for c in AUDIO_CLASSES}
for _, y in audio_items:
    counts[AUDIO_CLASSES[y]] += 1

print("=" * 60)
print("AUDIO — REAL FILE INVENTORY")
print("=" * 60)
print("source:", audio_source_info.get("source"))
for c in AUDIO_CLASSES:
    print(f"  {c:15}: {counts[c]}")
print(f"  {'TOTAL':15}: {len(audio_items)}")

_ys = [y for _, y in audio_items]
_min = min(counts.values())
if _min < 2:
    raise RuntimeError("At least 2 real clips per class are required. Class counts: " + str(counts))

audio_train, audio_val = train_test_split(
    audio_items, test_size=AUDIO_VAL_FRACTION, random_state=SEED, stratify=_ys)

_tr = {str(p.resolve()).lower() for p, _ in audio_train}
audio_val = [(p, y) for p, y in audio_val if str(p.resolve()).lower() not in _tr]

write_sample_manifest(AUDIO_RUN_DIR, "train", audio_train)
write_sample_manifest(AUDIO_RUN_DIR, "val", audio_val)

audio_split_info = {
    **audio_source_info,
    "split": f"stratified in-notebook split (seed={SEED}, val={AUDIO_VAL_FRACTION})",
    "train_size": len(audio_train),
    "val_size": len(audio_val),
    "class_counts": counts,
}
print("\nTrain / Val:", len(audio_train), "/", len(audio_val))
print("Sample real files:")
for p, y in audio_train[:5]:
    print(f"  [{AUDIO_CLASSES[y]:14}] {p}")

Manifest could not be parsed: download_manifest.csv: no path-like column found in ['ytid', 'start', 'end', 'label']
AUDIO — REAL FILE INVENTORY
source: directory_scan
  speech_normal  : 160
  siren          : 156
  fire           : 105
  scream         : 55
  TOTAL          : 476

Train / Val: 380 / 96
Sample real files:
  [speech_normal ] G:\Capstone\data\audio_data\audioset_clips\speech_normal\0X6CXKUbX6g_30.0.wav
  [siren         ] G:\Capstone\data\audio_data\audioset_clips\siren\lnClssYHVsE_30.0.wav
  [speech_normal ] G:\Capstone\data\audio_data\audioset_clips\speech_normal\-6x2PtSRfJU_30.0.wav
  [siren         ] G:\Capstone\data\audio_data\audioset_clips\siren\bDNz4NDWzaM_70.0.wav
  [siren         ] G:\Capstone\data\audio_data\audioset_clips\siren\J8Kx1I1B5V4_30.0.wav


## CELL 19 — Real WAV loader → log-Mel

```text
REAL WAV → load waveform → 16 kHz → mono → 4-second crop/pad → Log-Mel Spectrogram → CNN
```

Longer than 4 s ⇒ a crop is taken **from the real waveform** (random during training,
centre during validation so evaluation is deterministic). Shorter ⇒ zero padding.
No synthetic clip is ever created.

In [18]:
# ---- drop unusable clips BEFORE splitting ----
import soundfile as sf

MIN_AUDIO_SECONDS = 0.25      # the pipeline zero-pads to 4s, so short-but-real clips are fine

def _usable(p):
    if p.name.startswith("._"):
        return False, "macOS AppleDouble stub"
    if p.suffix.lower() != ".wav":
        return False, f"not a wav ({p.suffix})"
    try:
        info = sf.info(str(p))
    except Exception as e:
        return False, f"unreadable ({type(e).__name__})"
    if info.frames == 0:
        return False, "zero length"
    if info.duration < MIN_AUDIO_SECONDS:
        return False, f"shorter than {MIN_AUDIO_SECONDS}s"
    return True, "ok"

kept, dropped = [], {}
for p, y in audio_items:
    ok, why = _usable(p)
    (kept if ok else dropped.setdefault(why, [])).append((p, y) if ok else p)

print("AUDIO FILE FILTER")
print(f"  kept    : {len(kept)} / {len(audio_items)}")
for why, ps in sorted(dropped.items(), key=lambda kv: -len(kv[1])):
    print(f"  dropped : {len(ps):>4}   {why:<28} e.g. {ps[0].name}")

audio_items = kept
_after = {c: 0 for c in AUDIO_CLASSES}
for _, y in audio_items:
    _after[AUDIO_CLASSES[y]] += 1
print("\n  surviving per class:", _after)
for c, n in _after.items():
    if n < 40:
        print(f"  WARNING: '{c}' has only {n} clips - per-class metrics will be unreliable")

AUDIO FILE FILTER
  kept    : 375 / 476
  dropped :   92   zero length                  e.g. -8n2NqDFRko_30.0.wav
  dropped :    7   not a wav (.m4a)             e.g. -30H9V1IKps_6.0.temp.m4a
  dropped :    2   unreadable (LibsndfileError) e.g. -MZx1np1Ddc_30.0.wav

  surviving per class: {'speech_normal': 121, 'siren': 127, 'fire': 86, 'scream': 41}


In [ ]:
# ============================================================
# CELL 19 — Real WAV loader -> log-Mel   (COMPLETE REPLACEMENT)
# ============================================================
import soundfile as sf_probe
from sklearn.model_selection import train_test_split

# ---------- PRE-FLIGHT: drop unusable clips, rebuild the split ----------
MIN_AUDIO_SECONDS = 0.25      # pipeline zero-pads to 4s, so short-but-real clips are fine

def _audio_usable(p):
    p = Path(p)
    if p.name.startswith("._"):
        return False, "macOS AppleDouble stub"
    if p.suffix.lower() != ".wav":
        return False, f"not a wav ({p.suffix})"
    try:
        info = sf_probe.info(str(p))
    except Exception as e:
        return False, f"unreadable ({type(e).__name__})"
    if info.frames == 0:
        return False, "zero length"
    if info.duration < MIN_AUDIO_SECONDS:
        return False, f"under {MIN_AUDIO_SECONDS}s"
    return True, "ok"

_kept, _dropped = [], {}
for _p, _yy in audio_items:
    ok, why = _audio_usable(_p)
    if ok:
        _kept.append((_p, _yy))
    else:
        _dropped.setdefault(why, []).append(Path(_p).name)

print("=" * 60)
print("AUDIO PRE-FLIGHT FILTER")
print("=" * 60)
print(f"kept {len(_kept)} / {len(audio_items)}")
for why, names in sorted(_dropped.items(), key=lambda kv: -len(kv[1])):
    print(f"  dropped {len(names):>4}   {why:<26} e.g. {names[0]}")

audio_items = _kept
_counts = {c: 0 for c in AUDIO_CLASSES}
for _p, _yy in audio_items:
    _counts[AUDIO_CLASSES[_yy]] += 1
print("\nsurviving per class:", _counts)
for c, n in _counts.items():
    if n < 40:
        print(f"  WARNING: '{c}' has only {n} clips - per-class metrics will be unreliable")
if min(_counts.values()) < 2:
    raise RuntimeError("A class has fewer than 2 usable clips: " + str(_counts))

_ys = [y for _, y in audio_items]
audio_train, audio_val = train_test_split(
    audio_items, test_size=AUDIO_VAL_FRACTION, random_state=SEED, stratify=_ys)
_tr = {str(p.resolve()).lower() for p, _ in audio_train}
audio_val = [(p, y) for p, y in audio_val if str(p.resolve()).lower() not in _tr]

write_sample_manifest(AUDIO_RUN_DIR, "train", audio_train)
write_sample_manifest(AUDIO_RUN_DIR, "val", audio_val)
audio_split_info.update({
    "filtered_out": {k: len(v) for k, v in _dropped.items()},
    "min_audio_seconds": MIN_AUDIO_SECONDS,
    "train_size": len(audio_train), "val_size": len(audio_val),
    "class_counts": _counts,
})
print(f"\nrebuilt split: train {len(audio_train)} | val {len(audio_val)}")

# ---------- backends ----------
try:
    import torchaudio
    _HAS_TORCHAUDIO = True
except Exception:
    torchaudio = None
    _HAS_TORCHAUDIO = False

try:
    import librosa
    _HAS_LIBROSA = True
except Exception:
    librosa = None
    _HAS_LIBROSA = False

try:
    import soundfile as sf
    _HAS_SF = True
except Exception:
    sf = None
    _HAS_SF = False

AUDIO_LOAD_BACKEND = ("torchaudio" if _HAS_TORCHAUDIO else
                      "librosa" if _HAS_LIBROSA else
                      "soundfile" if _HAS_SF else None)
if AUDIO_LOAD_BACKEND is None:
    raise RuntimeError("Install torchaudio (preferred), librosa or soundfile to read real WAV files.")

AUDIO_MEL_BACKEND = ("torchaudio" if _HAS_TORCHAUDIO else
                     "librosa" if _HAS_LIBROSA else "torch")
print("\naudio load backend:", AUDIO_LOAD_BACKEND, "| mel backend:", AUDIO_MEL_BACKEND)


def load_waveform(path, target_sr=AUDIO_SAMPLE_RATE):
    # REAL audio file -> float32 mono numpy at target_sr. Never returns an empty array.
    path_obj = assert_real_file(path)
    spath, wav = str(path_obj), None

    if _HAS_TORCHAUDIO:
        try:
            w, sr = torchaudio.load(spath)
            if w.numel() == 0:
                raise ValueError("empty audio stream")
            if w.shape[0] > 1:
                w = w.mean(dim=0, keepdim=True)
            if sr != target_sr:
                w = torchaudio.functional.resample(w, sr, target_sr)
            wav = w.squeeze(0).numpy().astype(np.float32)
        except Exception:
            wav = None

    if wav is None and _HAS_LIBROSA:
        try:
            y, _ = librosa.load(spath, sr=target_sr, mono=True)
            wav = np.asarray(y, dtype=np.float32)
        except Exception:
            wav = None

    if wav is None and _HAS_SF:
        try:
            y, sr = sf.read(spath, dtype="float32", always_2d=True)
            y = y.mean(axis=1)
            if sr != target_sr and len(y) > 1:
                n = int(round(len(y) * target_sr / sr))
                y = np.interp(np.linspace(0, len(y) - 1, n), np.arange(len(y)), y)
            wav = np.asarray(y, dtype=np.float32)
        except Exception:
            wav = None

    if wav is None or wav.size == 0:
        raise ValueError(f"no decodable audio in {path_obj.name}")
    return wav


def fix_length(wav, n_samples=AUDIO_N_SAMPLES, random_crop=False, rng=None):
    # Crop from the REAL waveform, or zero-pad. Never fabricates signal content.
    wav = np.asarray(wav, dtype=np.float32)
    if wav.size == 0:
        raise ValueError("empty waveform reached fix_length")
    if len(wav) == n_samples:
        return wav
    if len(wav) > n_samples:
        if random_crop:
            r = rng or random
            start = r.randint(0, len(wav) - n_samples)
        else:
            start = max(0, (len(wav) - n_samples) // 2)
        return wav[start:start + n_samples]
    pad = n_samples - len(wav)
    left = pad // 2
    return np.pad(wav, (left, pad - left), mode="constant")


if _HAS_TORCHAUDIO:
    _MEL = torchaudio.transforms.MelSpectrogram(
        sample_rate=AUDIO_SAMPLE_RATE, n_fft=AUDIO_N_FFT, hop_length=AUDIO_HOP_LENGTH,
        n_mels=AUDIO_N_MELS, f_min=AUDIO_F_MIN, f_max=AUDIO_F_MAX, power=2.0)
    _DB = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=AUDIO_TOP_DB)


def _hz_to_mel(f):
    return 2595.0 * np.log10(1.0 + np.asarray(f, dtype=np.float64) / 700.0)


def _mel_to_hz(m):
    return 700.0 * (10.0 ** (np.asarray(m, dtype=np.float64) / 2595.0) - 1.0)


def _build_mel_filterbank(sr, n_fft, n_mels, fmin, fmax):
    fft_freqs = np.linspace(0, sr / 2.0, n_fft // 2 + 1)
    pts = _mel_to_hz(np.linspace(_hz_to_mel(fmin), _hz_to_mel(fmax), n_mels + 2))
    fb = np.zeros((n_mels, len(fft_freqs)), dtype=np.float32)
    for i in range(n_mels):
        lo, ctr, hi = pts[i], pts[i + 1], pts[i + 2]
        left = (fft_freqs - lo) / max(ctr - lo, 1e-9)
        right = (hi - fft_freqs) / max(hi - ctr, 1e-9)
        fb[i] = np.maximum(0.0, np.minimum(left, right))
        if hi - lo > 0:
            fb[i] *= 2.0 / (hi - lo)
    return torch.from_numpy(fb)


_TORCH_FB = None
_TORCH_WIN = None

def _log_mel_torch(wav):
    global _TORCH_FB, _TORCH_WIN
    if _TORCH_FB is None:
        _TORCH_FB = _build_mel_filterbank(AUDIO_SAMPLE_RATE, AUDIO_N_FFT,
                                          AUDIO_N_MELS, AUDIO_F_MIN, AUDIO_F_MAX)
        _TORCH_WIN = torch.hann_window(AUDIO_N_FFT)
    x = torch.from_numpy(np.ascontiguousarray(wav)).float()
    spec = torch.stft(x, n_fft=AUDIO_N_FFT, hop_length=AUDIO_HOP_LENGTH,
                      window=_TORCH_WIN, center=True, pad_mode="reflect",
                      return_complex=True).abs() ** 2
    mel = _TORCH_FB @ spec
    db = 10.0 * torch.log10(torch.clamp(mel, min=1e-10))
    db = torch.clamp(db, min=float(db.max()) - AUDIO_TOP_DB)
    return db.numpy().astype(np.float32)


def log_mel(wav):
    # REAL waveform -> log-mel spectrogram [n_mels, T]
    if AUDIO_MEL_BACKEND == "torchaudio":
        t = torch.from_numpy(np.ascontiguousarray(wav)).float()
        return _DB(_MEL(t)).numpy()
    if AUDIO_MEL_BACKEND == "librosa":
        S = librosa.feature.melspectrogram(
            y=wav, sr=AUDIO_SAMPLE_RATE, n_fft=AUDIO_N_FFT, hop_length=AUDIO_HOP_LENGTH,
            n_mels=AUDIO_N_MELS, fmin=AUDIO_F_MIN, fmax=AUDIO_F_MAX, power=2.0)
        return librosa.power_to_db(S, ref=np.max, top_db=AUDIO_TOP_DB).astype(np.float32)
    return _log_mel_torch(wav)


def normalize_spec(spec):
    m, s = float(spec.mean()), float(spec.std())
    return (spec - m) / (s + 1e-5)


# ---------- sanity check across several REAL files ----------
print("\n" + "=" * 60)
print("LOADER CHECK")
print("=" * 60)
for _p, _y in audio_train[:5]:
    _w = load_waveform(_p)
    _s = normalize_spec(log_mel(fix_length(_w, random_crop=False)))
    print(f"  [{AUDIO_CLASSES[_y]:14}] {len(_w)/AUDIO_SAMPLE_RATE:5.2f}s -> {_s.shape}  "
          f"[{_s.min():.2f} .. {_s.max():.2f}]  {Path(_p).name[:40]}")
    assert _s.shape[0] == AUDIO_N_MELS
print(f"\nexpected spectrogram shape: ({AUDIO_N_MELS}, {AUDIO_TIME_FRAMES})")

audio load backend: torchaudio | mel backend: torchaudio


RuntimeError: cannot reshape tensor of 0 elements into shape [-1, 0] because the unspecified dimension size -1 can be any value and is ambiguous

## CELL 20 — Real audio augmentation

```text
real waveform
   ├── small time shift
   ├── amplitude variation
   ├── real environmental noise (mixed in from ANOTHER REAL recording)
   └── spectrogram masking (SpecAugment on the real spectrogram)
```

No generated gunshot / scream spectrograms.

In [20]:
class RealAudioDataset(Dataset):
    # Every item is computed from a real audio file. `noise_pool` holds real recordings
    # used for background-noise mixing - it is never synthetic noise.
    def __init__(self, items, augment=False, noise_pool=None):
        self.items = items
        self.augment = augment
        self.noise_pool = noise_pool or []
        self.rng = random.Random(SEED)

    def __len__(self):
        return len(self.items)

    def _augment_wave(self, wav):
        # 1) time shift of the REAL waveform
        if self.rng.random() < 0.5:
            shift = self.rng.randint(-AUDIO_SAMPLE_RATE // 4, AUDIO_SAMPLE_RATE // 4)
            wav = np.roll(wav, shift)
        # 2) amplitude / gain variation
        if self.rng.random() < 0.5:
            wav = wav * self.rng.uniform(0.7, 1.3)
        # 3) mix in REAL background audio from another real file
        if self.noise_pool and self.rng.random() < 0.3:
            npath = self.rng.choice(self.noise_pool)
            try:
                nwav = fix_length(load_waveform(npath), random_crop=True, rng=self.rng)
                snr_db = self.rng.uniform(10.0, 25.0)
                ps = float(np.mean(wav ** 2)) + 1e-9
                pn = float(np.mean(nwav ** 2)) + 1e-9
                scale = math.sqrt(ps / (pn * (10 ** (snr_db / 10.0))))
                wav = wav + scale * nwav
            except Exception:
                pass
        peak = float(np.max(np.abs(wav))) if len(wav) else 0.0
        if peak > 1.0:
            wav = wav / peak
        return wav.astype(np.float32)

    def _spec_augment(self, spec):
        # SpecAugment applied to the REAL spectrogram
        spec = spec.copy()
        n_mels, n_t = spec.shape
        for _ in range(self.rng.randint(0, 2)):
            f = self.rng.randint(1, max(2, n_mels // 8))
            f0 = self.rng.randint(0, max(0, n_mels - f))
            spec[f0:f0 + f, :] = spec.mean()
        for _ in range(self.rng.randint(0, 2)):
            t = self.rng.randint(1, max(2, n_t // 8))
            t0 = self.rng.randint(0, max(0, n_t - t))
            spec[:, t0:t0 + t] = spec.mean()
        return spec

    def __getitem__(self, idx):
        path, label = self.items[idx]
        try:
            wav = load_waveform(path)
        except Exception:
            alt = self.items[(idx + 1) % len(self.items)]
            wav, label = load_waveform(alt[0]), alt[1]
        wav = fix_length(wav, random_crop=self.augment, rng=self.rng)
        if self.augment:
            wav = self._augment_wave(wav)
        spec = log_mel(wav)
        if spec.shape[1] < AUDIO_TIME_FRAMES:
            spec = np.pad(spec, ((0, 0), (0, AUDIO_TIME_FRAMES - spec.shape[1])),
                          mode="edge")
        spec = spec[:, :AUDIO_TIME_FRAMES]
        if self.augment:
            spec = self._spec_augment(spec)
        spec = normalize_spec(spec)
        return torch.from_numpy(spec).unsqueeze(0).float(), label


# Real recordings used as the background-noise pool (ambient/normal speech only)
NOISE_POOL = [p for p, y in audio_train if AUDIO_CLASSES[y] == "speech_normal"]
print("Real background-noise pool size:", len(NOISE_POOL))

audio_train_ds = RealAudioDataset(audio_train, augment=True, noise_pool=NOISE_POOL)
audio_val_ds   = RealAudioDataset(audio_val, augment=False)

_x, _y = audio_train_ds[0]
print("augmented sample tensor:", tuple(_x.shape), "label:", AUDIO_CLASSES[_y])

Real background-noise pool size: 128
augmented sample tensor: (1, 64, 251) label: siren


## CELL 21 — Train the audio CNN on real spectrograms

Output: `audio_classifier_real_v2.pt`, carrying the exact preprocessing contract the
SENTRIX inference code must reuse.

In [22]:
# ============================================================
# AUDIO REPAIR — run this cell, then re-run the training cell
# ============================================================
from sklearn.model_selection import train_test_split

MIN_AUDIO_SECONDS = 0.25

# ---- 1. hardened loader (overrides the one defined earlier) ----
def load_waveform(path, target_sr=AUDIO_SAMPLE_RATE):
    path_obj = assert_real_file(path)
    spath, wav = str(path_obj), None

    if _HAS_TORCHAUDIO:
        try:
            w, sr = torchaudio.load(spath)
            if w.numel() == 0:
                raise ValueError("empty audio stream")
            if w.shape[0] > 1:
                w = w.mean(dim=0, keepdim=True)
            if sr != target_sr:
                w = torchaudio.functional.resample(w, sr, target_sr)
            wav = w.squeeze(0).numpy().astype(np.float32)
        except Exception:
            wav = None

    if wav is None and _HAS_LIBROSA:
        try:
            wav = np.asarray(librosa.load(spath, sr=target_sr, mono=True)[0], dtype=np.float32)
        except Exception:
            wav = None

    if wav is None and _HAS_SF:
        try:
            y, sr = sf.read(spath, dtype="float32", always_2d=True)
            y = y.mean(axis=1)
            if sr != target_sr and len(y) > 1:
                n = int(round(len(y) * target_sr / sr))
                y = np.interp(np.linspace(0, len(y) - 1, n), np.arange(len(y)), y)
            wav = np.asarray(y, dtype=np.float32)
        except Exception:
            wav = None

    if wav is None or wav.size == 0:
        raise ValueError(f"no decodable audio in {path_obj.name}")
    return wav


# ---- 2. actually decode every candidate once; keep only what loads ----
print("=" * 60)
print("AUDIO REPAIR")
print("=" * 60)
_min_samples = int(MIN_AUDIO_SECONDS * AUDIO_SAMPLE_RATE)
kept, dropped = [], {}
for p, y in audio_items:
    p = Path(p)
    if p.name.startswith("._"):
        dropped.setdefault("AppleDouble stub", []).append(p.name); continue
    if p.suffix.lower() != ".wav":
        dropped.setdefault(f"not a wav ({p.suffix})", []).append(p.name); continue
    try:
        w = load_waveform(p)
    except Exception as e:
        dropped.setdefault(f"undecodable ({type(e).__name__})", []).append(p.name); continue
    if w.size < _min_samples:
        dropped.setdefault(f"under {MIN_AUDIO_SECONDS}s", []).append(p.name); continue
    kept.append((p, y))

print(f"decoded OK : {len(kept)} / {len(audio_items)}")
for why, names in sorted(dropped.items(), key=lambda kv: -len(kv[1])):
    print(f"  dropped {len(names):>4}   {why:<30} e.g. {names[0]}")
audio_items = kept

# ---- 3. rebuild split, noise pool and BOTH datasets ----
counts = {c: 0 for c in AUDIO_CLASSES}
for _, y in audio_items:
    counts[AUDIO_CLASSES[y]] += 1
print("\nper class:", counts)
for c, n in counts.items():
    if n < 40:
        print(f"  WARNING: '{c}' has only {n} clips - treat its F1 as unreportable")
if min(counts.values()) < 2:
    raise RuntimeError("class with fewer than 2 usable clips: " + str(counts))

_ys = [y for _, y in audio_items]
audio_train, audio_val = train_test_split(
    audio_items, test_size=AUDIO_VAL_FRACTION, random_state=SEED, stratify=_ys)
_tr = {str(p.resolve()).lower() for p, _ in audio_train}
audio_val = [(p, y) for p, y in audio_val if str(p.resolve()).lower() not in _tr]

write_sample_manifest(AUDIO_RUN_DIR, "train", audio_train)
write_sample_manifest(AUDIO_RUN_DIR, "val", audio_val)
audio_split_info.update({
    "filtered_out": {k: len(v) for k, v in dropped.items()},
    "min_audio_seconds": MIN_AUDIO_SECONDS,
    "train_size": len(audio_train), "val_size": len(audio_val),
    "class_counts": counts,
})

NOISE_POOL = [p for p, y in audio_train if AUDIO_CLASSES[y] == "speech_normal"]
audio_train_ds = RealAudioDataset(audio_train, augment=True, noise_pool=NOISE_POOL)
audio_val_ds   = RealAudioDataset(audio_val, augment=False)

print(f"\nrebuilt: train {len(audio_train)} | val {len(audio_val)} | noise pool {len(NOISE_POOL)}")
_x, _y = audio_train_ds[0]
print("sample tensor:", tuple(_x.shape), "| label:", AUDIO_CLASSES[_y])
print("\n>>> now re-run the AUDIO TRAINING cell <<<")

AUDIO REPAIR
decoded OK : 375 / 375

per class: {'speech_normal': 121, 'siren': 127, 'fire': 86, 'scream': 41}

rebuilt: train 300 | val 75 | noise pool 97
sample tensor: (1, 64, 251) | label: speech_normal

>>> now re-run the AUDIO TRAINING cell <<<


In [23]:
class SentrixAudioCNN(nn.Module):
    # Log-Mel CNN. Input: (B, 1, n_mels, T)
    def __init__(self, num_classes, n_mels=AUDIO_N_MELS):
        super().__init__()

        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1, bias=False),
                nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                nn.Conv2d(cout, cout, 3, padding=1, bias=False),
                nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )

        self.features = nn.Sequential(
            block(1, 32), block(32, 64), block(64, 128), block(128, 256),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))


def train_audio_model():
    set_seed(SEED)
    train_ld = DataLoader(audio_train_ds, batch_size=AUDIO_BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
    val_ld   = DataLoader(audio_val_ds, batch_size=AUDIO_BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))

    model = SentrixAudioCNN(len(AUDIO_CLASSES)).to(DEVICE)
    weights = class_weights_from_items(audio_train, len(AUDIO_CLASSES)).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)
    optimizer = torch.optim.AdamW(model.parameters(), lr=AUDIO_LR,
                                  weight_decay=AUDIO_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=AUDIO_EPOCHS)
    scaler = make_scaler()

    print("=" * 60)
    print("TRAINING: AUDIO (real WAV -> log-mel)")
    print("=" * 60)
    print("Train / Val :", len(audio_train), "/", len(audio_val))
    print("Class weights:", [round(float(w), 3) for w in weights.cpu()])

    ckpt = AUDIO_RUN_DIR / "best_checkpoint.pt"
    history, best_f1, best_epoch, bad = [], -1.0, -1, 0
    t_start = time.time()

    for epoch in range(1, AUDIO_EPOCHS + 1):
        model.train()
        running, seen, correct = 0.0, 0, 0
        t0 = time.time()
        for xb, yb in train_ld:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with amp_ctx():
                out = model(xb)
                loss = criterion(out, yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            running += loss.item() * xb.size(0)
            seen += xb.size(0)
            correct += (out.float().argmax(1) == yb).sum().item()
        scheduler.step()

        y_true, y_prob, y_pred = evaluate_classifier(model, val_ld, len(AUDIO_CLASSES))
        m = classification_report_dict(y_true, y_pred, y_prob, AUDIO_CLASSES)
        history.append({"epoch": epoch, "train_loss": running / max(seen, 1),
                        "train_acc": correct / max(seen, 1),
                        "val_acc": m["accuracy"], "val_macro_f1": m["macro_f1"],
                        "seconds": time.time() - t0})
        flag = ""
        if m["macro_f1"] > best_f1 + 1e-5:
            best_f1, best_epoch, bad = m["macro_f1"], epoch, 0
            torch.save({"model_state_dict": model.state_dict(),
                        "classes": AUDIO_CLASSES, "epoch": epoch,
                        "val_macro_f1": best_f1}, ckpt)
            flag = "  <-- best (checkpoint saved)"
        else:
            bad += 1
        print(f"epoch {epoch:>3}/{AUDIO_EPOCHS} | loss {history[-1]['train_loss']:.4f} | "
              f"train_acc {history[-1]['train_acc']:.4f} | val_acc {m['accuracy']:.4f} | "
              f"val_macroF1 {m['macro_f1']:.4f} | {history[-1]['seconds']:.1f}s{flag}")
        if bad >= AUDIO_PATIENCE:
            print(f"Early stopping at epoch {epoch}.")
            break

    ck = torch.load(ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck["model_state_dict"])
    y_true, y_prob, y_pred = evaluate_classifier(model, val_ld, len(AUDIO_CLASSES))
    final = classification_report_dict(y_true, y_pred, y_prob, AUDIO_CLASSES)
    final["best_epoch"] = best_epoch
    final["epochs_run"] = len(history)
    final["training_seconds"] = round(time.time() - t_start, 1)
    print_metrics("AUDIO", final)

    # --- production checkpoint: weights + the exact preprocessing contract ---
    torch.save({
        "classes": AUDIO_CLASSES,
        "sample_rate": AUDIO_SAMPLE_RATE,
        "duration_seconds": AUDIO_DURATION,
        "n_samples": AUDIO_N_SAMPLES,
        "n_mels": AUDIO_N_MELS,
        "n_fft": AUDIO_N_FFT,
        "hop_length": AUDIO_HOP_LENGTH,
        "f_min": AUDIO_F_MIN,
        "f_max": AUDIO_F_MAX,
        "top_db": AUDIO_TOP_DB,
        "power": 2.0,
        "amplitude_to_db": True,
        "normalization": "per_sample_mean_std",
        "input_shape": [1, AUDIO_N_MELS, AUDIO_TIME_FRAMES],
        "channels": "mono",
        "architecture": "SentrixAudioCNN",
        "model_state_dict": model.state_dict(),
        "val_macro_f1": final["macro_f1"],
        "run_id": RUN_ID,
        "training_type": "real_data",
        "synthetic_data": False,
    }, AUDIO_MODEL_V3)
    print("\nModel saved to:", AUDIO_MODEL_V3)

    pd.DataFrame(history).to_csv(AUDIO_RUN_DIR / "history.csv", index=False)
    (AUDIO_RUN_DIR / "metrics.json").write_text(
        json.dumps(final, indent=2, default=str), encoding="utf-8")

    meta_path = save_metadata(AUDIO_MODEL_V3, {
        "model_name": AUDIO_MODEL_V3.stem,
        "architecture": "SentrixAudioCNN (log-mel CNN)",
        "dataset": str(AUDIO_DATA),
        "manifest": str(AUDIO_MANIFEST) if AUDIO_MANIFEST.exists() else "directory_scan",
        "classes": AUDIO_CLASSES,
        "class_index_map": {c: i for i, c in enumerate(AUDIO_CLASSES)},
        "sample_rate": AUDIO_SAMPLE_RATE,
        "duration_seconds": AUDIO_DURATION,
        "n_mels": AUDIO_N_MELS,
        "n_fft": AUDIO_N_FFT,
        "hop_length": AUDIO_HOP_LENGTH,
        "f_min": AUDIO_F_MIN,
        "f_max": AUDIO_F_MAX,
        "top_db": AUDIO_TOP_DB,
        "normalization": "per_sample_mean_std",
        "input_shape": [1, AUDIO_N_MELS, AUDIO_TIME_FRAMES],
        "mel_backend": AUDIO_MEL_BACKEND,
        "augmentation": ["time_shift", "gain", "real_background_noise_mix", "spec_augment"],
        "split_info": audio_split_info,
        "metrics": {k: final[k] for k in
                    ["accuracy", "macro_f1", "macro_precision", "macro_recall",
                     "per_class", "confusion_matrix"] if k in final},
        "training_type": "real_data",
        "synthetic_data": False,
    })
    return model, final, meta_path


try:
    audio_model, audio_metrics, audio_meta_path = train_audio_model()
    RESULTS["audio"] = audio_metrics
    ARTIFACTS["audio"] = {"model": AUDIO_MODEL_V3, "metadata": audio_meta_path,
                          "run_dir": AUDIO_RUN_DIR}
    MODULE_STATUS["audio"] = "TRAINED"
except Exception as e:
    MODULE_STATUS["audio"] = "FAILED"
    print("AUDIO TRAINING FAILED:", type(e).__name__, e)
    raise

TRAINING: AUDIO (real WAV -> log-mel)
Train / Val : 300 / 75
Class weights: [0.773, 0.743, 1.087, 2.273]
epoch   1/25 | loss 1.3161 | train_acc 0.4433 | val_acc 0.2667 | val_macroF1 0.2363 | 3.2s  <-- best (checkpoint saved)
epoch   2/25 | loss 1.1482 | train_acc 0.5867 | val_acc 0.4533 | val_macroF1 0.3975 | 2.3s  <-- best (checkpoint saved)
epoch   3/25 | loss 1.0767 | train_acc 0.6033 | val_acc 0.4533 | val_macroF1 0.4495 | 2.5s  <-- best (checkpoint saved)
epoch   4/25 | loss 0.9808 | train_acc 0.6767 | val_acc 0.5600 | val_macroF1 0.4427 | 2.4s
epoch   5/25 | loss 0.9997 | train_acc 0.6433 | val_acc 0.5867 | val_macroF1 0.5107 | 2.6s  <-- best (checkpoint saved)
epoch   6/25 | loss 0.9609 | train_acc 0.6733 | val_acc 0.6533 | val_macroF1 0.6068 | 2.3s  <-- best (checkpoint saved)
epoch   7/25 | loss 0.9192 | train_acc 0.6633 | val_acc 0.6400 | val_macroF1 0.6239 | 2.5s  <-- best (checkpoint saved)
epoch   8/25 | loss 0.9364 | train_acc 0.6967 | val_acc 0.4667 | val_macroF1 0.4330 

---
---
# MODULES 3 & 4 — YOLO DETECTORS (weapon, fire/smoke)

Both detectors use the real YOLO datasets shipped with `data.yaml`.
Unlike the previous pipeline, **validation is enabled during training** (`val=True`)
and the runs are long enough to converge.

## CELL 21b — YOLO helpers

Resolves `data.yaml` to absolute paths under `G:\Capstone\data` (Roboflow exports usually
carry relative or stale paths), verifies the split directories exist, and never fabricates images.

In [24]:
def prepare_yolo_yaml(src_yaml: Path, dataset_root: Path, run_dir: Path, expected_classes):
    # Returns (patched_yaml_path, info). Rewrites train/val/test to absolute REAL directories.
    import yaml as _yaml
    if not src_yaml.exists():
        raise RuntimeError(f"data.yaml not found: {src_yaml}")
    cfg = _yaml.safe_load(src_yaml.read_text(encoding="utf-8"))

    names = cfg.get("names")
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names, key=lambda x: int(x))]
    names = list(names or [])

    base = cfg.get("path")
    base_dir = (dataset_root / str(base)).resolve() if base and not Path(str(base)).is_absolute() \
        else (Path(str(base)).resolve() if base else dataset_root)
    if not base_dir.exists():
        base_dir = dataset_root

    def resolve_split(val):
        if val is None:
            return None
        vals = val if isinstance(val, list) else [val]
        out = []
        for v in vals:
            v = str(v).replace("\\", "/")
            for cand in [Path(v), base_dir / v, dataset_root / v,
                         dataset_root / Path(v).name,
                         dataset_root / "images" / Path(v).name]:
                if cand.exists():
                    out.append(str(cand.resolve()))
                    break
        return out[0] if len(out) == 1 else (out or None)

    patched = {
        "path": str(dataset_root.resolve()),
        "train": resolve_split(cfg.get("train")),
        "val": resolve_split(cfg.get("val") or cfg.get("valid")),
        "names": names,
        "nc": cfg.get("nc", len(names)),
    }
    test = resolve_split(cfg.get("test"))
    if test:
        patched["test"] = test

    # Last-resort discovery if the yaml pointed nowhere real
    if not patched["train"]:
        for cand in [dataset_root / "train" / "images", dataset_root / "images" / "train",
                     dataset_root / "train"]:
            if cand.exists():
                patched["train"] = str(cand.resolve())
                break
    if not patched["val"]:
        for cand in [dataset_root / "valid" / "images", dataset_root / "val" / "images",
                     dataset_root / "images" / "val", dataset_root / "valid",
                     dataset_root / "val"]:
            if cand.exists():
                patched["val"] = str(cand.resolve())
                break

    if not patched["train"] or not patched["val"]:
        raise RuntimeError(
            f"Could not resolve real train/val image directories for {dataset_root}. "
            f"yaml said train={cfg.get('train')} val={cfg.get('val')}. "
            "No data will be generated - fix the dataset layout."
        )

    out_yaml = run_dir / "data_v2_real.yaml"
    out_yaml.write_text(_yaml.safe_dump(patched, sort_keys=False), encoding="utf-8")

    info = {
        "source_yaml": str(src_yaml),
        "patched_yaml": str(out_yaml),
        "classes_in_yaml": names,
        "expected_classes": list(expected_classes),
        "class_names_match": [n.lower() for n in names] == [c.lower() for c in expected_classes],
        "train": patched["train"],
        "val": patched["val"],
    }
    return out_yaml, info


def train_yolo(name, data_yaml: Path, epochs, imgsz, batch, patience,
               run_dir: Path, out_model: Path, dataset_dir: Path,
               yaml_info: dict, base_weights="yolov8n.pt"):
    from ultralytics import YOLO
    set_seed(SEED)

    print("=" * 60)
    print(f"TRAINING: {name} (YOLO)")
    print("=" * 60)
    print("data.yaml :", data_yaml)
    print("classes   :", yaml_info["classes_in_yaml"])
    if not yaml_info["class_names_match"]:
        print("NOTE: yaml class order differs from the expected SENTRIX order",
              yaml_info["expected_classes"],
              "- the class IDs recorded in the metadata are the authoritative mapping.")
    print("epochs    :", epochs, "| imgsz:", imgsz, "| batch:", batch, "| patience:", patience)
    print("validation during training: ENABLED")

    device = 0 if DEVICE == "cuda" else "cpu"
    model = YOLO(base_weights)
    t0 = time.time()
    model.train(
        data=str(data_yaml),
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        patience=patience,
        project=str(run_dir),
        name="train",
        exist_ok=True,
        device=device,
        seed=SEED,
        val=True,              # <-- never disabled
        plots=True,
        workers=NUM_WORKERS,
        pretrained=True,
        verbose=True,
    )
    train_seconds = time.time() - t0

    best = run_dir / "train" / "weights" / "best.pt"
    if not best.exists():
        cands = sorted(run_dir.rglob("best.pt"))
        if not cands:
            raise RuntimeError(f"{name}: YOLO produced no best.pt under {run_dir}")
        best = cands[-1]

    # Explicit validation pass on the real validation split
    val_model = YOLO(str(best))
    vr = val_model.val(data=str(data_yaml), imgsz=imgsz, device=device,
                       project=str(run_dir), name="val", exist_ok=True, plots=True)

    box = vr.box
    metrics = {
        "mAP50": float(box.map50),
        "mAP50_95": float(box.map),
        "precision": float(box.mp),
        "recall": float(box.mr),
        "per_class_mAP50": {},
        "training_seconds": round(train_seconds, 1),
        "weights": str(best),
    }
    try:
        names = val_model.names
        for i, ap in enumerate(box.maps):
            metrics["per_class_mAP50"][str(names.get(i, i))] = float(ap)
    except Exception:
        pass

    print("\n" + "=" * 60)
    print(f"{name} — VALIDATION METRICS")
    print("=" * 60)
    print(f"mAP50     : {metrics['mAP50']:.4f}")
    print(f"mAP50-95  : {metrics['mAP50_95']:.4f}")
    print(f"Precision : {metrics['precision']:.4f}")
    print(f"Recall    : {metrics['recall']:.4f}")
    for k, v in metrics["per_class_mAP50"].items():
        print(f"   {k:12}: {v:.4f}")

    shutil.copy2(best, out_model)
    print("\nModel saved to:", out_model)

    (run_dir / "metrics.json").write_text(
        json.dumps(metrics, indent=2, default=str), encoding="utf-8")

    meta_path = save_metadata(out_model, {
        "model_name": out_model.stem,
        "architecture": f"YOLOv8 ({base_weights})",
        "dataset": str(dataset_dir),
        "data_yaml": str(data_yaml),
        "classes": yaml_info["classes_in_yaml"],
        "class_id_map": {i: n for i, n in enumerate(yaml_info["classes_in_yaml"])},
        "image_size": imgsz,
        "batch_size": batch,
        "epochs_configured": epochs,
        "patience": patience,
        "validation_during_training": True,
        "seed": SEED,
        "device": str(device),
        "recommended_conf_threshold": 0.25,
        "recommended_iou_threshold": 0.45,
        "metrics": metrics,
        "training_type": "real_data",
        "synthetic_data": False,
    })
    return metrics, meta_path


print("YOLO helpers ready.")

YOLO helpers ready.


---
# MODULE 3 — WEAPON

## CELL 22 — Weapon configuration

Classes: `knife`, `long_gun`, `pistol`

In [25]:
WEAPON_YAML       = WEAPON_DATA / "data.yaml"
WEAPON_EPOCHS     = 50
WEAPON_IMAGE_SIZE = 640
WEAPON_BATCH_SIZE = 16
WEAPON_PATIENCE   = 10
WEAPON_CLASSES    = ["knife", "long_gun", "pistol"]
WEAPON_BASE_WEIGHTS = "yolov8n.pt"    # use yolov8s.pt / yolov8m.pt if the GPU allows

WEAPON_RUN_DIR = V2_RUNS_DIR / "weapon"
WEAPON_RUN_DIR.mkdir(parents=True, exist_ok=True)

print("WEAPON dataset :", WEAPON_DATA)
print("data.yaml      :", WEAPON_YAML, "| exists:", WEAPON_YAML.exists())
if not WEAPON_YAML.exists():
    for c in sorted(WEAPON_DATA.glob("*.yaml")) + sorted(WEAPON_DATA.glob("*.yml")):
        print("   candidate yaml found:", c)
print("run dir        :", WEAPON_RUN_DIR)
print("output model   :", WEAPON_MODEL_V3)

WEAPON dataset : G:\Capstone\data\weapon_data
data.yaml      : G:\Capstone\data\weapon_data\data.yaml | exists: True
run dir        : G:\Sentrix\training\runs_v3_real\weapon
output model   : G:\Sentrix\backend\models\v3_real\weapon_detector_real_v3.pt


## CELL 22b — Train the weapon detector

In [26]:
try:
    _wy = WEAPON_YAML if WEAPON_YAML.exists() else (
        (sorted(WEAPON_DATA.glob("*.yaml")) + sorted(WEAPON_DATA.glob("*.yml")))[0])
    weapon_yaml_v2, weapon_yaml_info = prepare_yolo_yaml(
        _wy, WEAPON_DATA, WEAPON_RUN_DIR, WEAPON_CLASSES)
    print(json.dumps(weapon_yaml_info, indent=2))

    weapon_metrics, weapon_meta_path = train_yolo(
        name="WEAPON",
        data_yaml=weapon_yaml_v2,
        epochs=WEAPON_EPOCHS,
        imgsz=WEAPON_IMAGE_SIZE,
        batch=WEAPON_BATCH_SIZE,
        patience=WEAPON_PATIENCE,
        run_dir=WEAPON_RUN_DIR,
        out_model=WEAPON_MODEL_V3,
        dataset_dir=WEAPON_DATA,
        yaml_info=weapon_yaml_info,
        base_weights=WEAPON_BASE_WEIGHTS,
    )
    RESULTS["weapon"] = weapon_metrics
    ARTIFACTS["weapon"] = {"model": WEAPON_MODEL_V3, "metadata": weapon_meta_path,
                           "run_dir": WEAPON_RUN_DIR}
    MODULE_STATUS["weapon"] = "TRAINED"
except Exception as e:
    MODULE_STATUS["weapon"] = "FAILED"
    print("WEAPON TRAINING FAILED:", type(e).__name__, e)
    raise

{
  "source_yaml": "G:\\Capstone\\data\\weapon_data\\data.yaml",
  "patched_yaml": "G:\\Sentrix\\training\\runs_v3_real\\weapon\\data_v2_real.yaml",
  "classes_in_yaml": [
    "knife",
    "long_gun",
    "pistol"
  ],
  "expected_classes": [
    "knife",
    "long_gun",
    "pistol"
  ],
  "class_names_match": true,
  "train": "G:\\Capstone\\data\\weapon_data\\train\\images",
  "val": "G:\\Capstone\\data\\weapon_data\\valid\\images"
}
TRAINING: WEAPON (YOLO)
data.yaml : G:\Sentrix\training\runs_v3_real\weapon\data_v2_real.yaml
classes   : ['knife', 'long_gun', 'pistol']
epochs    : 50 | imgsz: 640 | batch: 16 | patience: 10
validation during training: ENABLED
New https://pypi.org/project/ultralytics/8.4.127 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.203  Python-3.11.15 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=Fa

---
# MODULE 4 — FIRE / SMOKE

## CELL 23 — Fire/smoke configuration

Classes: `fire`, `smoke`

In [27]:
FIRE_SMOKE_YAML       = FIRE_SMOKE_DATA / "data.yaml"
FIRE_SMOKE_EPOCHS     = 50
FIRE_SMOKE_IMAGE_SIZE = 640
FIRE_SMOKE_BATCH_SIZE = 16
FIRE_SMOKE_PATIENCE   = 10
FIRE_SMOKE_CLASSES    = ["fire", "smoke"]
FIRE_SMOKE_BASE_WEIGHTS = "yolov8n.pt"

FIRE_SMOKE_RUN_DIR = V2_RUNS_DIR / "fire_smoke"
FIRE_SMOKE_RUN_DIR.mkdir(parents=True, exist_ok=True)

print("FIRE/SMOKE dataset :", FIRE_SMOKE_DATA)
print("data.yaml          :", FIRE_SMOKE_YAML, "| exists:", FIRE_SMOKE_YAML.exists())
if not FIRE_SMOKE_YAML.exists():
    for c in sorted(FIRE_SMOKE_DATA.glob("*.yaml")) + sorted(FIRE_SMOKE_DATA.glob("*.yml")):
        print("   candidate yaml found:", c)
print("run dir            :", FIRE_SMOKE_RUN_DIR)
print("output model       :", FIRE_SMOKE_MODEL_V3)

FIRE/SMOKE dataset : G:\Capstone\data\fire_smoke_data
data.yaml          : G:\Capstone\data\fire_smoke_data\data.yaml | exists: True
run dir            : G:\Sentrix\training\runs_v3_real\fire_smoke
output model       : G:\Sentrix\backend\models\v3_real\fire_smoke_detector_real_v3.pt


## CELL 23b — Train the fire/smoke detector

In [28]:
try:
    _fy = FIRE_SMOKE_YAML if FIRE_SMOKE_YAML.exists() else (
        (sorted(FIRE_SMOKE_DATA.glob("*.yaml")) + sorted(FIRE_SMOKE_DATA.glob("*.yml")))[0])
    fire_yaml_v2, fire_yaml_info = prepare_yolo_yaml(
        _fy, FIRE_SMOKE_DATA, FIRE_SMOKE_RUN_DIR, FIRE_SMOKE_CLASSES)
    print(json.dumps(fire_yaml_info, indent=2))

    fire_metrics, fire_meta_path = train_yolo(
        name="FIRE_SMOKE",
        data_yaml=fire_yaml_v2,
        epochs=FIRE_SMOKE_EPOCHS,
        imgsz=FIRE_SMOKE_IMAGE_SIZE,
        batch=FIRE_SMOKE_BATCH_SIZE,
        patience=FIRE_SMOKE_PATIENCE,
        run_dir=FIRE_SMOKE_RUN_DIR,
        out_model=FIRE_SMOKE_MODEL_V3,
        dataset_dir=FIRE_SMOKE_DATA,
        yaml_info=fire_yaml_info,
        base_weights=FIRE_SMOKE_BASE_WEIGHTS,
    )
    RESULTS["fire_smoke"] = fire_metrics
    ARTIFACTS["fire_smoke"] = {"model": FIRE_SMOKE_MODEL_V3, "metadata": fire_meta_path,
                               "run_dir": FIRE_SMOKE_RUN_DIR}
    MODULE_STATUS["fire_smoke"] = "TRAINED"
except Exception as e:
    MODULE_STATUS["fire_smoke"] = "FAILED"
    print("FIRE/SMOKE TRAINING FAILED:", type(e).__name__, e)
    raise

{
  "source_yaml": "G:\\Capstone\\data\\fire_smoke_data\\data.yaml",
  "patched_yaml": "G:\\Sentrix\\training\\runs_v3_real\\fire_smoke\\data_v2_real.yaml",
  "classes_in_yaml": [
    "fire",
    "smoke"
  ],
  "expected_classes": [
    "fire",
    "smoke"
  ],
  "class_names_match": true,
  "train": "G:\\Capstone\\data\\fire_smoke_data\\train\\images",
  "val": "G:\\Capstone\\data\\fire_smoke_data\\valid\\images"
}
TRAINING: FIRE_SMOKE (YOLO)
data.yaml : G:\Sentrix\training\runs_v3_real\fire_smoke\data_v2_real.yaml
classes   : ['fire', 'smoke']
epochs    : 50 | imgsz: 640 | batch: 16 | patience: 10
validation during training: ENABLED
New https://pypi.org/project/ultralytics/8.4.127 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.203  Python-3.11.15 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=Non

---
---
# MODULE 5 — REAL MULTIMODAL FUSION (TCI)

## The single most important rule in this notebook

**The notebook must NOT manufacture the fusion dataset.**

The previous fusion trainer built synthetic multimodal scenarios with synthetic TCI targets.
That is exactly what is being eliminated. Module 5 runs **only** if a real, labelled,
event-level dataset exists on disk. Otherwise it stops and tells you what to collect.

## CELL 24 — Fusion data discovery

Searches `G:\Capstone\data` and `G:\Sentrix\data` for a real event-level dataset such as
`fusion.csv`, `multimodal.csv`, `events.csv`, `tci_dataset.csv`, `threat_events.csv`.

In [29]:
FUSION_SEARCH_ROOTS = [DATA_ROOT, PROJECT_ROOT / "data", TRAINING_DIR / "data"]
FUSION_NAME_HINTS = ["fusion", "multimodal", "events", "tci", "threat", "incident"]

# `anomaly_score` is produced by the UCF-Crime-DVS notebook
# (sentrix_dvs_infer -> rec["anomaly_score"]), NOT by an image classifier in this notebook.
FUSION_FEATURE_CANDIDATES = [
    "violence_score", "anomaly_score", "audio_score", "weapon_score",
    "fire_score", "motion_score", "identity_score",
]
FUSION_TARGET_CANDIDATES = [
    "ground_truth_tci", "tci", "tci_score", "target_tci", "true_tci",
]
FUSION_LEVEL_CANDIDATES = ["ground_truth_level", "level", "threat_level", "severity"]

# Manifests that are NOT event-level fusion data
FUSION_EXCLUDE = {"train.csv", "val.csv", "test.csv", "download_manifest.csv",
                  "balanced_train_segments.csv", "eval_segments.csv",
                  "unbalanced_train_segments.csv", "class_labels_indices.csv"}


def scan_fusion_candidates():
    found = []
    for root in FUSION_SEARCH_ROOTS:
        if not root.exists():
            continue
        for p in root.rglob("*.csv"):
            if p.name.lower() in FUSION_EXCLUDE:
                continue
            try:
                head = pd.read_csv(p, nrows=5)
            except Exception:
                continue
            cols = [str(c).strip().lower() for c in head.columns]
            feats = [c for c in FUSION_FEATURE_CANDIDATES if c in cols]
            target = next((c for c in FUSION_TARGET_CANDIDATES if c in cols), None)
            level = next((c for c in FUSION_LEVEL_CANDIDATES if c in cols), None)
            name_hit = any(h in p.name.lower() for h in FUSION_NAME_HINTS)
            if (len(feats) >= 3 and (target or level)) or (name_hit and (target or level)):
                found.append({"path": p, "n_features": len(feats), "features": feats,
                              "target": target, "level": level, "columns": cols})
    found.sort(key=lambda d: (d["target"] is not None, d["n_features"]), reverse=True)
    return found


print("=" * 60)
print("FUSION DATA DISCOVERY")
print("=" * 60)
for r in FUSION_SEARCH_ROOTS:
    print("searching:", r, "| exists:", r.exists())

fusion_candidates = scan_fusion_candidates()
FUSION_DATASET = None

if fusion_candidates:
    print(f"\n{len(fusion_candidates)} candidate dataset(s) found:")
    for c in fusion_candidates:
        print(f"  {c['path']}")
        print(f"     features: {c['features']}")
        print(f"     target  : {c['target']}   level: {c['level']}")
    FUSION_DATASET = fusion_candidates[0]
    print("\nSELECTED:", FUSION_DATASET["path"])
else:
    print()
    print("=" * 60)
    print("FUSION TRAINING NOT EXECUTED")
    print("=" * 60)
    print("Reason:")
    print("No real multimodal ground-truth dataset was found.")
    print("Synthetic data generation is disabled by design.")
    print("Create/collect real labelled SENTRIX events before")
    print("training the fusion model.")
    print("=" * 60)
    print()
    print("Expected columns in such a dataset:")
    for c in ["event_id"] + FUSION_FEATURE_CANDIDATES + ["ground_truth_tci",
                                                         "ground_truth_level", "incident_type"]:
        print("   -", c)
    MODULE_STATUS["fusion"] = "SKIPPED"

FUSION DATA DISCOVERY
searching: G:\Capstone\data | exists: True
searching: G:\Sentrix\data | exists: False
searching: G:\Sentrix\training\data | exists: False

FUSION TRAINING NOT EXECUTED
Reason:
No real multimodal ground-truth dataset was found.
Synthetic data generation is disabled by design.
Create/collect real labelled SENTRIX events before
training the fusion model.

Expected columns in such a dataset:
   - event_id
   - violence_score
   - anomaly_score
   - audio_score
   - weapon_score
   - fire_score
   - motion_score
   - identity_score
   - ground_truth_tci
   - ground_truth_level
   - incident_type


## CELL 25 — Fusion feature configuration

Runs only if CELL 24 found a real dataset. No random generation, no manually invented target.

In [30]:
FUSION_FEATURES = None
FUSION_TARGET = None
fusion_df = None

if FUSION_DATASET is None:
    print("Module 5 skipped — no real event dataset. Nothing is generated.")
else:
    fusion_df = pd.read_csv(FUSION_DATASET["path"])
    fusion_df.columns = [str(c).strip().lower() for c in fusion_df.columns]

    FUSION_FEATURES = [c for c in FUSION_FEATURE_CANDIDATES if c in fusion_df.columns]
    FUSION_TARGET = FUSION_DATASET["target"]
    FUSION_LEVEL = FUSION_DATASET["level"]

    print("Dataset :", FUSION_DATASET["path"])
    print("Rows    :", len(fusion_df))
    print("Features:", FUSION_FEATURES)
    print("Target  :", FUSION_TARGET)
    print("Level   :", FUSION_LEVEL)

    if FUSION_TARGET is None:
        raise RuntimeError(
            "A real event dataset was found but it has no numeric TCI target column "
            f"(one of {FUSION_TARGET_CANDIDATES}). Label the real events before training. "
            "No target will be manufactured."
        )
    if len(FUSION_FEATURES) < 3:
        raise RuntimeError(
            f"Only {len(FUSION_FEATURES)} usable feature columns found: {FUSION_FEATURES}. "
            "Fusion needs real per-modality scores. No features will be manufactured."
        )

    before = len(fusion_df)
    fusion_df = fusion_df.dropna(subset=FUSION_FEATURES + [FUSION_TARGET])
    print(f"Rows after dropping incomplete real events: {len(fusion_df)} (dropped {before-len(fusion_df)})")

    X = fusion_df[FUSION_FEATURES]
    y = fusion_df[FUSION_TARGET]
    print("\nFeature summary (real values only):")
    print(X.describe().T[["count", "mean", "std", "min", "max"]])
    print("\nTarget summary:")
    print(y.describe())

Module 5 skipped — no real event dataset. Nothing is generated.


## CELL 26 — XGBoost fusion training

Output: `tci_xgboost_real_v3.json`

In [31]:
FUSION_RUN_DIR = V2_RUNS_DIR / "fusion"
FUSION_RUN_DIR.mkdir(parents=True, exist_ok=True)

if FUSION_DATASET is None:
    print("=" * 60)
    print("FUSION TRAINING NOT EXECUTED")
    print("=" * 60)
    print("Reason:")
    print("No real multimodal ground-truth dataset was found.")
    print("Synthetic data generation is disabled by design.")
    print("Create/collect real labelled SENTRIX events before")
    print("training the fusion model.")
    print("=" * 60)
    MODULE_STATUS["fusion"] = "SKIPPED"
else:
    import xgboost as xgb

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=SEED)

    print("Train / Val events:", len(X_train), "/", len(X_val))

    xgb_params = dict(
        n_estimators=600,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        min_child_weight=2,
        random_state=SEED,
        n_jobs=-1,
        early_stopping_rounds=50,
        eval_metric="rmse",
        tree_method="hist",
        device="cuda" if DEVICE == "cuda" else "cpu",
    )
    try:
        fusion_model = xgb.XGBRegressor(**xgb_params)
        fusion_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    except TypeError:
        # older xgboost: no `device` / constructor-level early stopping
        xgb_params.pop("device", None)
        xgb_params.pop("early_stopping_rounds", None)
        fusion_model = xgb.XGBRegressor(**xgb_params)
        fusion_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    pred = fusion_model.predict(X_val)
    fusion_metrics = {
        "MAE": float(mean_absolute_error(y_val, pred)),
        "RMSE": float(math.sqrt(mean_squared_error(y_val, pred))),
        "R2": float(r2_score(y_val, pred)),
        "tci_correlation": float(np.corrcoef(np.asarray(y_val, dtype=float), pred)[0, 1]),
        "n_train": int(len(X_train)),
        "n_val": int(len(X_val)),
        "features": FUSION_FEATURES,
        "target": FUSION_TARGET,
        "feature_importance": {f: float(v) for f, v in
                               zip(FUSION_FEATURES, fusion_model.feature_importances_)},
    }

    # High/Critical behaviour, using the real target scale
    scale_max = float(np.nanmax(np.asarray(y, dtype=float)))
    high_cut = 0.70 * scale_max
    yv = np.asarray(y_val, dtype=float)
    true_high = yv >= high_cut
    pred_high = pred >= high_cut
    tp = int(np.sum(true_high & pred_high)); fp = int(np.sum(~true_high & pred_high))
    fn = int(np.sum(true_high & ~pred_high))
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    fusion_metrics["high_critical_threshold"] = high_cut
    fusion_metrics["high_critical_precision"] = float(prec)
    fusion_metrics["high_critical_recall"] = float(rec)
    fusion_metrics["high_critical_f1"] = float(2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
    fusion_metrics["false_negative_rate"] = float(fn / (fn + tp)) if (fn + tp) else 0.0

    print("\n" + "=" * 60)
    print("FUSION — VALIDATION METRICS (real events)")
    print("=" * 60)
    for k in ["MAE", "RMSE", "R2", "tci_correlation", "high_critical_f1", "false_negative_rate"]:
        print(f"{k:22}: {fusion_metrics[k]:.4f}")
    print("\nFeature importance:")
    for f, v in sorted(fusion_metrics["feature_importance"].items(), key=lambda x: -x[1]):
        print(f"  {f:18}: {v:.4f}")

    fusion_model.save_model(str(FUSION_MODEL_V3))
    print("\nModel saved to:", FUSION_MODEL_V3)

    (FUSION_RUN_DIR / "metrics.json").write_text(
        json.dumps(fusion_metrics, indent=2, default=str), encoding="utf-8")

    fusion_meta_path = save_metadata(FUSION_MODEL_V3, {
        "model_name": FUSION_MODEL_V3.stem,
        "architecture": "XGBoost regressor",
        "dataset": str(FUSION_DATASET["path"]),
        "features": FUSION_FEATURES,
        "feature_order": FUSION_FEATURES,
        "target": FUSION_TARGET,
        "target_scale_max": scale_max,
        "seed": SEED,
        "metrics": fusion_metrics,
        "training_type": "real_data",
        "synthetic_data": False,
    })

    RESULTS["fusion"] = fusion_metrics
    ARTIFACTS["fusion"] = {"model": FUSION_MODEL_V3, "metadata": fusion_meta_path,
                           "run_dir": FUSION_RUN_DIR}
    MODULE_STATUS["fusion"] = "TRAINED"

FUSION TRAINING NOT EXECUTED
Reason:
No real multimodal ground-truth dataset was found.
Synthetic data generation is disabled by design.
Create/collect real labelled SENTRIX events before
training the fusion model.


---
---
# CELL 27 — Consolidated model evaluation

One standardised report per module, written to `runs_v2_real/EVALUATION_REPORT.md`.

In [32]:
lines = []
def emit(s=""):
    print(s)
    lines.append(s)

emit("=" * 70)
emit("SENTRIX V3 REAL-DATA — CONSOLIDATED EVALUATION")
emit("=" * 70)
emit(f"Run ID   : {RUN_ID}")
emit(f"Data root: {DATA_ROOT}")
emit(f"Models   : {V3_MODELS_DIR}")
emit("")

emit("Module status")
emit("-" * 40)
for k, v in MODULE_STATUS.items():
    emit(f"  {k:12}: {v}")
emit("")

for mod in ["violence"]:
    m = RESULTS.get(mod)
    if not m:
        continue
    emit(f"{mod.upper()}")
    emit("-" * 40)
    emit(f"  Accuracy            : {m['accuracy']:.4f}")
    emit(f"  Precision (macro)   : {m['macro_precision']:.4f}")
    emit(f"  Recall (macro)      : {m['macro_recall']:.4f}")
    emit(f"  F1 (macro)          : {m['macro_f1']:.4f}")
    if m.get("pr_auc") is not None:
        emit(f"  PR-AUC              : {m['pr_auc']:.4f}")
    if "false_negative_rate" in m:
        emit(f"  False Negative Rate : {m['false_negative_rate']:.4f}")
        emit(f"  Operating threshold : {m['best_threshold']:.3f}")
    emit(f"  Confusion matrix    : {m['confusion_matrix']}")
    emit("")

m = RESULTS.get("audio")
if m:
    emit("AUDIO")
    emit("-" * 40)
    emit(f"  Accuracy   : {m['accuracy']:.4f}")
    emit(f"  Macro F1   : {m['macro_f1']:.4f}")
    for c, d in m["per_class"].items():
        emit(f"    {c:15} P {d['precision']:.3f}  R {d['recall']:.3f}  "
             f"F1 {d['f1']:.3f}  n={d['support']}")
    emit(f"  Confusion matrix: {m['confusion_matrix']}")
    emit("")

for mod in ["weapon", "fire_smoke"]:
    m = RESULTS.get(mod)
    if not m:
        continue
    emit(f"{mod.upper()}")
    emit("-" * 40)
    emit(f"  mAP50     : {m['mAP50']:.4f}")
    emit(f"  mAP50-95  : {m['mAP50_95']:.4f}")
    emit(f"  Precision : {m['precision']:.4f}")
    emit(f"  Recall    : {m['recall']:.4f}")
    for c, v in m.get("per_class_mAP50", {}).items():
        emit(f"    {c:12}: {v:.4f}")
    emit("")

m = RESULTS.get("fusion")
if m:
    emit("FUSION (TCI)")
    emit("-" * 40)
    emit(f"  MAE                 : {m['MAE']:.4f}")
    emit(f"  RMSE                : {m['RMSE']:.4f}")
    emit(f"  R2                  : {m['R2']:.4f}")
    emit(f"  TCI correlation     : {m['tci_correlation']:.4f}")
    emit(f"  High/Critical F1    : {m['high_critical_f1']:.4f}")
    emit(f"  False Negative Rate : {m['false_negative_rate']:.4f}")
    emit("")
elif MODULE_STATUS["fusion"] == "SKIPPED":
    emit("FUSION (TCI)")
    emit("-" * 40)
    emit("  NOT TRAINED - no real labelled multimodal event dataset was found.")
    emit("  Synthetic data generation is disabled by design.")
    emit("")

report_path = V2_RUNS_DIR / "EVALUATION_REPORT.md"
report_path.write_text("```text\n" + "\n".join(lines) + "\n```\n", encoding="utf-8")
(V2_RUNS_DIR / f"results_{RUN_ID}.json").write_text(
    json.dumps(RESULTS, indent=2, default=str), encoding="utf-8")
print("Report saved:", report_path)

SENTRIX V3 REAL-DATA — CONSOLIDATED EVALUATION
Run ID   : 20260824_193729
Data root: G:\Capstone\data
Models   : G:\Sentrix\backend\models\v3_real

Module status
----------------------------------------
  violence    : TRAINED
  audio       : TRAINED
  weapon      : TRAINED
  fire_smoke  : TRAINED
  fusion      : SKIPPED

VIOLENCE
----------------------------------------
  Accuracy            : 0.8326
  Precision (macro)   : 0.8340
  Recall (macro)      : 0.8314
  F1 (macro)          : 0.8319
  PR-AUC              : 0.8827
  False Negative Rate : 0.1325
  Operating threshold : 0.398
  Confusion matrix    : [[1511, 389], [269, 1761]]

AUDIO
----------------------------------------
  Accuracy   : 0.7200
  Macro F1   : 0.7036
    speech_normal   P 0.700  R 0.875  F1 0.778  n=24
    siren           P 0.933  R 0.538  F1 0.683  n=26
    fire            P 0.684  R 0.765  F1 0.722  n=17
    scream          P 0.545  R 0.750  F1 0.632  n=8
  Confusion matrix: [[21, 0, 2, 1], [4, 14, 4, 4], [4, 0

---
# CELL 28 — Production preprocessing verification

```text
TRAINING PREPROCESSING   vs   SENTRIX APPLICATION PREPROCESSING
```

Prints the contract every inference path must honour, writes it to
`v2_real/PRODUCTION_PREPROCESSING_CONTRACT.json`, and scans `G:\Sentrix\backend`
for values that disagree with it.

In [33]:
CONTRACT = {
    "images_violence": {
        "resolution": [VIOLENCE_IMAGE_SIZE, VIOLENCE_IMAGE_SIZE],
        "color_space": "RGB (convert from OpenCV BGR before inference)",
        "scaling": "ToTensor -> [0,1]",
        "normalization_mean": [0.485, 0.456, 0.406],
        "normalization_std": [0.229, 0.224, 0.225],
        "tensor_layout": "NCHW float32",
        "violence_class_order": VIOLENCE_CLASSES,
        "positive_index_violence": VIOLENCE_CLASSES.index("fight"),
    },
    "anomaly": {
        "owner": "SENTRIX_UCF_CRIME_DVS_TRAINING.ipynb (event / DVS)",
        "model": str(UCF_DVS_MODEL),
        "input": "UCF-Crime-DVS .npz event data - NOT RGB images",
        "consumed_by_fusion": "anomaly_score only",
    },
    "audio": {
        "sample_rate": AUDIO_SAMPLE_RATE,
        "channels": "mono",
        "clip_seconds": AUDIO_DURATION,
        "n_samples": AUDIO_N_SAMPLES,
        "n_mels": AUDIO_N_MELS,
        "n_fft": AUDIO_N_FFT,
        "hop_length": AUDIO_HOP_LENGTH,
        "f_min": AUDIO_F_MIN,
        "f_max": AUDIO_F_MAX,
        "power": 2.0,
        "amplitude_to_db": True,
        "top_db": AUDIO_TOP_DB,
        "normalization": "per-sample (x - mean) / (std + 1e-5)",
        "input_shape": [1, AUDIO_N_MELS, AUDIO_TIME_FRAMES],
        "class_order": AUDIO_CLASSES,
        "long_clip_policy": "centre crop at inference; random crop only during training",
        "short_clip_policy": "symmetric zero padding",
    },
    "yolo": {
        "image_size": WEAPON_IMAGE_SIZE,
        "weapon_classes": WEAPON_CLASSES,
        "fire_smoke_classes": FIRE_SMOKE_CLASSES,
        "recommended_conf": 0.25,
        "recommended_iou": 0.45,
        "letterbox": "handled internally by ultralytics - pass the raw BGR/RGB frame",
    },
    "fusion": {
        "feature_order": FUSION_FEATURES if FUSION_FEATURES else "NOT TRAINED",
        "target": FUSION_TARGET if FUSION_TARGET else "NOT TRAINED",
    },
}

contract_path = V3_MODELS_DIR / "PRODUCTION_PREPROCESSING_CONTRACT.json"
contract_path.write_text(json.dumps(CONTRACT, indent=2, default=str), encoding="utf-8")

print("=" * 70)
print("PRODUCTION PREPROCESSING CONTRACT")
print("=" * 70)
print(json.dumps(CONTRACT, indent=2, default=str))
print("\nSaved to:", contract_path)

# --- scan the backend for values that disagree ---
CHECKS = [
    ("sample_rate",  r"sample_rate\s*=\s*(\d+)",        AUDIO_SAMPLE_RATE),
    ("sr",           r"\bsr\s*=\s*(\d+)",               AUDIO_SAMPLE_RATE),
    ("n_mels",       r"n_mels\s*=\s*(\d+)",             AUDIO_N_MELS),
    ("n_fft",        r"n_fft\s*=\s*(\d+)",              AUDIO_N_FFT),
    ("hop_length",   r"hop_length\s*=\s*(\d+)",         AUDIO_HOP_LENGTH),
    ("imgsz",        r"imgsz\s*=\s*(\d+)",              WEAPON_IMAGE_SIZE),
]

import re
print("\n" + "=" * 70)
print("BACKEND SCAN — values that differ from the training contract")
print("=" * 70)
mismatches = []
py_files = list(BACKEND_DIR.rglob("*.py")) if BACKEND_DIR.exists() else []
print(f"Scanned {len(py_files)} python files under {BACKEND_DIR}")
for f in py_files:
    try:
        text = f.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue
    for label, pattern, expected in CHECKS:
        for mt in re.finditer(pattern, text):
            val = int(mt.group(1))
            if val != expected:
                line_no = text[:mt.start()].count("\n") + 1
                mismatches.append((str(f), line_no, label, val, expected))

if mismatches:
    for f, ln, label, val, exp in mismatches[:60]:
        print(f"  MISMATCH {label:12} = {val:<8} (training uses {exp})  {f}:{ln}")
    print(f"\n{len(mismatches)} potential mismatch(es). Review before promoting any V2 model.")
else:
    print("  No conflicting values detected (or backend not reachable from this machine).")

# class-order references
print("\nClass-order references found in the backend:")
for f in py_files:
    try:
        text = f.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue
    for cls_list, name in [(AUDIO_CLASSES, "audio"), (VIOLENCE_CLASSES, "violence"),
                           (WEAPON_CLASSES, "weapon"), (FIRE_SMOKE_CLASSES, "fire_smoke")]:
        if all(c in text for c in cls_list):
            print(f"  {name:11} class names appear in {f}")

PRODUCTION PREPROCESSING CONTRACT
{
  "images_violence": {
    "resolution": [
      224,
      224
    ],
    "color_space": "RGB (convert from OpenCV BGR before inference)",
    "scaling": "ToTensor -> [0,1]",
    "normalization_mean": [
      0.485,
      0.456,
      0.406
    ],
    "normalization_std": [
      0.229,
      0.224,
      0.225
    ],
    "tensor_layout": "NCHW float32",
    "violence_class_order": [
      "nofight",
      "fight"
    ],
    "positive_index_violence": 1
  },
  "anomaly": {
    "owner": "SENTRIX_UCF_CRIME_DVS_TRAINING.ipynb (event / DVS)",
    "model": "G:\\Sentrix\\backend\\models\\v2_real\\ucf_crime_dvs\\ucf_crime_dvs_anomaly_v1.pt",
    "input": "UCF-Crime-DVS .npz event data - NOT RGB images",
    "consumed_by_fusion": "anomaly_score only"
  },
  "audio": {
    "sample_rate": 16000,
    "channels": "mono",
    "clip_seconds": 4.0,
    "n_samples": 64000,
    "n_mels": 64,
    "n_fft": 1024,
    "hop_length": 256,
    "f_min": 20,
    "f_max": 800

---
# CELL 29 — Model metadata index

Every V2 model has a `*_metadata.json` next to it. This cell verifies they exist and
writes a combined index.

In [34]:
expected_models = {
    "violence":   VIOLENCE_MODEL_V3,
    "audio":      AUDIO_MODEL_V3,
    "weapon":     WEAPON_MODEL_V3,
    "fire_smoke": FIRE_SMOKE_MODEL_V3,
    "fusion":     FUSION_MODEL_V3,
}

index = {"run_id": RUN_ID, "created": datetime.now().isoformat(timespec="seconds"),
         "project_root": str(PROJECT_ROOT), "data_root": str(DATA_ROOT),
         "training_type": "real_data", "synthetic_data": False, "models": {}}

print("=" * 70)
print("MODEL + METADATA INVENTORY")
print("=" * 70)
for mod, path in expected_models.items():
    meta = path.with_name(path.stem + "_metadata.json")
    entry = {
        "status": MODULE_STATUS.get(mod),
        "model_file": str(path),
        "model_exists": path.exists(),
        "model_size_mb": round(path.stat().st_size / 1024**2, 2) if path.exists() else None,
        "metadata_file": str(meta),
        "metadata_exists": meta.exists(),
    }
    index["models"][mod] = entry
    mark = "OK " if path.exists() and meta.exists() else ("-- " if MODULE_STATUS.get(mod) == "SKIPPED" else "!! ")
    print(f"{mark}{mod:11} {MODULE_STATUS.get(mod, ''):8} "
          f"model={'yes' if entry['model_exists'] else 'no ':4} "
          f"meta={'yes' if entry['metadata_exists'] else 'no ':4} "
          f"{entry['model_size_mb'] if entry['model_size_mb'] else ''}")

index["anomaly_module"] = {
    "owner_notebook": str(UCF_DVS_NOTEBOOK),
    "dataset": str(UCF_DVS_ROOT),
    "model": str(UCF_DVS_MODEL),
    "model_exists": UCF_DVS_MODEL.exists(),
    "note": "event-based; trained separately; supplies anomaly_score to fusion",
}

index_path = V3_MODELS_DIR / "MODEL_INDEX.json"
index_path.write_text(json.dumps(index, indent=2, default=str), encoding="utf-8")
print("\nIndex saved:", index_path)

MODEL + METADATA INVENTORY
OK violence    TRAINED  model=yes  meta=yes  42.72
OK audio       TRAINED  model=yes  meta=yes  4.63
OK weapon      TRAINED  model=yes  meta=yes  5.96
OK fire_smoke  TRAINED  model=yes  meta=yes  5.94
-- fusion      SKIPPED  model=no   meta=no   

Index saved: G:\Sentrix\backend\models\v3_real\MODEL_INDEX.json


---
# CELL 30 — Old vs new model comparison

The old models are **never deleted and never overwritten** — they are loaded read-only and
scored on the *same* V2 validation splits so the comparison is apples-to-apples.
A V2 model should only be promoted if it actually wins here **and** in application testing.

In [35]:
OLD_MODELS = {
    "violence":   OLD_MODELS_DIR / "violence_classifier.pt",
    "audio":      OLD_MODELS_DIR / "audio_classifier.pt",
    "weapon":     OLD_MODELS_DIR / "weapon_detector.pt",
    "fire_smoke": OLD_MODELS_DIR / "fire_smoke_detector.pt",
    "fusion":     OLD_MODELS_DIR / "tci_xgboost.json",
}

OLD_SCORES = {}


def _load_state_dict(obj):
    if isinstance(obj, dict):
        for k in ["model_state_dict", "state_dict", "model"]:
            if k in obj and isinstance(obj[k], dict):
                return obj[k], obj
        if all(isinstance(v, torch.Tensor) for v in obj.values()):
            return obj, {}
    return None, {}


def score_old_image_model(path: Path, val_items, classes, img_size):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    sd, extra = _load_state_dict(ck)
    if sd is None:
        raise RuntimeError("unrecognised checkpoint structure")
    old_classes = extra.get("classes", classes)
    model = build_resnet18(len(old_classes))
    try:
        model.load_state_dict(sd, strict=True)
    except Exception:
        # older heads were a plain Linear rather than Dropout+Linear
        model.fc = nn.Linear(model.fc[1].in_features, len(old_classes))
        model.load_state_dict(sd, strict=False)
    model.to(DEVICE)
    _, eval_tf = make_image_transforms(extra.get("image_size", img_size))
    ld = DataLoader(RealImageDataset(val_items, eval_tf), batch_size=32,
                    shuffle=False, num_workers=NUM_WORKERS)
    y_true, y_prob, y_pred = evaluate_classifier(model, ld, len(old_classes))
    return classification_report_dict(y_true, y_pred, y_prob, list(old_classes))


def score_old_audio_model(path: Path, val_items):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    sd, extra = _load_state_dict(ck)
    if sd is None:
        raise RuntimeError("unrecognised checkpoint structure")
    old_classes = extra.get("classes", AUDIO_CLASSES)
    model = SentrixAudioCNN(len(old_classes))
    model.load_state_dict(sd, strict=False)
    model.to(DEVICE)
    ds = RealAudioDataset(val_items, augment=False)
    ld = DataLoader(ds, batch_size=AUDIO_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    y_true, y_prob, y_pred = evaluate_classifier(model, ld, len(old_classes))
    return classification_report_dict(y_true, y_pred, y_prob, list(old_classes))


def score_old_yolo(path: Path, data_yaml: Path, imgsz):
    from ultralytics import YOLO
    m = YOLO(str(path))
    r = m.val(data=str(data_yaml), imgsz=imgsz,
              device=0 if DEVICE == "cuda" else "cpu", verbose=False)
    return {"mAP50": float(r.box.map50), "mAP50_95": float(r.box.map),
            "precision": float(r.box.mp), "recall": float(r.box.mr)}


print("=" * 70)
print("SCORING OLD MODELS ON THE V2 VALIDATION SPLITS (read-only)")
print("=" * 70)

for mod in ["violence", "audio", "weapon", "fire_smoke", "fusion"]:
    p = OLD_MODELS[mod]
    if not p.exists():
        print(f"  {mod:11}: old model not found ({p.name}) - nothing to compare")
        continue
    try:
        if mod == "violence":
            OLD_SCORES[mod] = score_old_image_model(p, violence_val, VIOLENCE_CLASSES,
                                                    VIOLENCE_IMAGE_SIZE)
        elif mod == "audio":
            OLD_SCORES[mod] = score_old_audio_model(p, audio_val)
        elif mod == "weapon":
            OLD_SCORES[mod] = score_old_yolo(p, weapon_yaml_v2, WEAPON_IMAGE_SIZE)
        elif mod == "fire_smoke":
            OLD_SCORES[mod] = score_old_yolo(p, fire_yaml_v2, FIRE_SMOKE_IMAGE_SIZE)
        elif mod == "fusion" and FUSION_DATASET is not None:
            import xgboost as xgb
            old = xgb.XGBRegressor()
            old.load_model(str(p))
            try:
                pr = old.predict(X_val[FUSION_FEATURES])
            except Exception:
                pr = old.predict(X_val)
            OLD_SCORES[mod] = {
                "RMSE": float(math.sqrt(mean_squared_error(y_val, pr))),
                "MAE": float(mean_absolute_error(y_val, pr)),
                "R2": float(r2_score(y_val, pr)),
            }
        print(f"  {mod:11}: scored OK")
    except Exception as e:
        print(f"  {mod:11}: could not be scored ({type(e).__name__}: {e})")

# ---------------- comparison table ----------------
def fmt(v):
    return f"{v:.4f}" if isinstance(v, (int, float)) else "n/a"


rows = []
def add_row(label, metric, old_v, new_v, higher_better=True):
    if isinstance(old_v, (int, float)) and isinstance(new_v, (int, float)):
        better = (new_v > old_v) if higher_better else (new_v < old_v)
        delta = new_v - old_v
        verdict = ("YES" if better else "no") + f"  ({delta:+.4f})"
    else:
        verdict = "cannot compare"
    rows.append((label, metric, fmt(old_v), fmt(new_v), verdict))


add_row("Violence", "F1 (macro)",
        OLD_SCORES.get("violence", {}).get("macro_f1"),
        RESULTS.get("violence", {}).get("macro_f1"))
add_row("Violence", "FNR",
        OLD_SCORES.get("violence", {}).get("false_negative_rate"),
        RESULTS.get("violence", {}).get("false_negative_rate"), higher_better=False)
add_row("Audio", "F1 (macro)",
        OLD_SCORES.get("audio", {}).get("macro_f1"),
        RESULTS.get("audio", {}).get("macro_f1"))
add_row("Weapon", "mAP50",
        OLD_SCORES.get("weapon", {}).get("mAP50"),
        RESULTS.get("weapon", {}).get("mAP50"))
add_row("Fire/Smoke", "mAP50",
        OLD_SCORES.get("fire_smoke", {}).get("mAP50"),
        RESULTS.get("fire_smoke", {}).get("mAP50"))
add_row("Fusion", "RMSE",
        OLD_SCORES.get("fusion", {}).get("RMSE"),
        RESULTS.get("fusion", {}).get("RMSE"), higher_better=False)

print("\n" + "=" * 78)
print("OLD vs NEW V3 (same validation data, old files untouched)")
print("=" * 78)
print(f"{'Model':<12}{'Metric':<14}{'Old':>10}{'New V3':>10}   Better?")
print("-" * 78)
for r in rows:
    print(f"{r[0]:<12}{r[1]:<14}{r[2]:>10}{r[3]:>10}   {r[4]}")

md_table = ["| Model | Metric | Old | New V3 | Better? |", "| --- | --- | ---: | ---: | --- |"]
md_table += [f"| {r[0]} | {r[1]} | {r[2]} | {r[3]} | {r[4]} |" for r in rows]
(V2_RUNS_DIR / "MODEL_COMPARISON.md").write_text("\n".join(md_table) + "\n", encoding="utf-8")
(V2_RUNS_DIR / f"old_scores_{RUN_ID}.json").write_text(
    json.dumps(OLD_SCORES, indent=2, default=str), encoding="utf-8")
print("\nSaved:", V2_RUNS_DIR / "MODEL_COMPARISON.md")
print("\nNOTE: 'cannot compare' simply means the old checkpoint could not be loaded or")
print("      was trained on a different label space. It is NOT evidence that V3 is better.")
print("\nANOMALY is intentionally absent from this table - the event model is evaluated")
print("in SENTRIX_UCF_CRIME_DVS_TRAINING.ipynb with AUC, not F1 on image frames.")

SCORING OLD MODELS ON THE V2 VALIDATION SPLITS (read-only)
  violence   : scored OK
  audio      : scored OK
Ultralytics 8.3.203  Python-3.11.15 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1.20.4 MB/s, size: 22.7 KB)
val: Scanning G:\Capstone\data\weapon_data\valid\labels.cache... 1599 images, 102 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1599/1599 1.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 100/100 7.3it/s 13.6s0.1s
                   all       1599       1666   3.36e-05    0.00793   1.73e-05   3.56e-06
Speed: 0.3ms preprocess, 3.0ms inference, 0.0ms loss, 2.6ms postprocess per image
Results saved to C:\Users\PC\runs\detect\val
  weapon     : scored OK
Ultralytics 8.3.203  Python-3.11.15 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB

---
# CELL 31 — Production model staging

Nothing under `backend/models/` is overwritten. All new artefacts live in
`backend/models/v3_real/`. Promotion is a **manual, deliberate** step performed only after
application testing — the helper below is provided but not called.

In [36]:
print("=" * 70)
print("OLD MODEL IMMUTABILITY CHECK")
print("=" * 70)
changed, missing_now = [], []
for name, snap in OLD_MODEL_SNAPSHOT.items():
    p = OLD_MODELS_DIR / name
    if not p.exists():
        missing_now.append(name)
        continue
    if "error" in snap:
        continue
    now = _fingerprint(p)
    if now["size"] != snap["size"] or now["sha256_head"] != snap["sha256_head"]:
        changed.append(name)

print(f"Tracked old model files : {len(OLD_MODEL_SNAPSHOT)}")
print(f"Modified by this run    : {len(changed)}")
print(f"Missing since snapshot  : {len(missing_now)}")
for n in changed:
    print("   MODIFIED:", n)
for n in missing_now:
    print("   MISSING :", n)
if not changed and not missing_now:
    print("\nAll previous SENTRIX models are byte-identical to the pre-run snapshot.")

print("\n" + "=" * 70)
print("V3 STAGING DIRECTORY")
print("=" * 70)
print(V3_MODELS_DIR)
for p in sorted(V3_MODELS_DIR.rglob("*")):
    if p.is_file():
        print(f"   {p.relative_to(V3_MODELS_DIR)}  ({p.stat().st_size/1024**2:.2f} MB)")


def promote_to_production(module_name, backup=True):
    # MANUAL STEP - call this yourself only after the V2 model has beaten the old one
    # in CELL 30 AND in real application testing.
    mapping = {
        "violence":   (VIOLENCE_MODEL_V3,   OLD_MODELS_DIR / "violence_classifier.pt"),
        "audio":      (AUDIO_MODEL_V3,      OLD_MODELS_DIR / "audio_classifier.pt"),
        "weapon":     (WEAPON_MODEL_V3,     OLD_MODELS_DIR / "weapon_detector.pt"),
        "fire_smoke": (FIRE_SMOKE_MODEL_V3, OLD_MODELS_DIR / "fire_smoke_detector.pt"),
        "fusion":     (FUSION_MODEL_V3,     OLD_MODELS_DIR / "tci_xgboost.json"),
    }
    if module_name not in mapping:
        raise ValueError(f"unknown module: {module_name}")
    src, dst = mapping[module_name]
    if not src.exists():
        raise FileNotFoundError(f"V2 model not found: {src}")
    if backup and dst.exists():
        bdir = OLD_MODELS_DIR / "archive_pre_v3"
        bdir.mkdir(parents=True, exist_ok=True)
        bpath = bdir / f"{dst.stem}_{RUN_ID}{dst.suffix}"
        shutil.copy2(dst, bpath)
        print("Backed up old model to:", bpath)
    shutil.copy2(src, dst)
    print("PROMOTED:", src.name, "->", dst)


print("\nPromotion helper defined but NOT executed.")
print("When you are ready:  promote_to_production('violence')")

OLD MODEL IMMUTABILITY CHECK
Tracked old model files : 7
Modified by this run    : 0
Missing since snapshot  : 0

All previous SENTRIX models are byte-identical to the pre-run snapshot.

V3 STAGING DIRECTORY
G:\Sentrix\backend\models\v3_real
   audio_classifier_real_v3.pt  (4.63 MB)
   audio_classifier_real_v3_metadata.json  (0.00 MB)
   fire_smoke_detector_real_v3.pt  (5.94 MB)
   fire_smoke_detector_real_v3_metadata.json  (0.00 MB)
   MODEL_INDEX.json  (0.00 MB)
   PRODUCTION_PREPROCESSING_CONTRACT.json  (0.00 MB)
   violence_classifier_real_v3.pt  (42.72 MB)
   violence_classifier_real_v3_metadata.json  (0.00 MB)
   weapon_detector_real_v3.pt  (5.96 MB)
   weapon_detector_real_v3_metadata.json  (0.00 MB)

Promotion helper defined but NOT executed.
When you are ready:  promote_to_production('violence')


---
# CELL 32 — Final verification

In [37]:
NOTEBOOK_PATH = TRAINING_DIR / "SENTRIX_REAL_DATA_RETRAINING_V3.ipynb"

checks = []

def check(label, ok, detail=""):
    checks.append((label, bool(ok), detail))


# 1. datasets
check("All real datasets found",
      all(p.exists() for p in [VIOLENCE_DATA, AUDIO_DATA, WEAPON_DATA, FIRE_SMOKE_DATA]),
      str(DATA_ROOT))
check("Anomaly delegated to the event notebook", True,
      f"{UCF_DVS_NOTEBOOK.name} | model present: {UCF_DVS_MODEL.exists()}")

# 2. no synthetic dataset classes / random-sample generation in this notebook's CODE cells
GENERATOR_CALLS = ["np.random.rand(", "np.random.randn(", "np.random.normal(",
                   "np.random.uniform(", "np.random.randint(", "torch.randn(",
                   "torch.rand(", "np.random.choice("]
banned_hits, generator_hits, scanned_cells = [], [], 0
if NOTEBOOK_PATH.exists():
    _nbjson = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8", errors="ignore"))
    for _i, _c in enumerate(_nbjson.get("cells", [])):
        if _c.get("cell_type") != "code":
            continue
        _src = "".join(_c.get("source", []))
        if "BANNED_SOURCE_MARKERS" in _src or "GENERATOR_CALLS" in _src:
            continue                      # the guard definitions themselves
        scanned_cells += 1
        for _mk in BANNED_SOURCE_MARKERS:
            if ("class " + _mk) in _src or (_mk + "(") in _src:
                banned_hits.append(f"cell {_i}: {_mk}")
        for _g in GENERATOR_CALLS:
            if _g in _src:
                generator_hits.append(f"cell {_i}: {_g}")

check("No synthetic dataset classes detected", not banned_hits,
      ("FOUND: " + ", ".join(banned_hits)) if banned_hits else
      (f"scanned {scanned_cells} code cells" if NOTEBOOK_PATH.exists()
       else "notebook file not found on disk"))
check("No random sample generation calls", not generator_hits,
      ("FOUND: " + ", ".join(generator_hits)) if generator_hits else
      "no array/tensor generators used to create samples")

# 3. no synthetic TCI
check("No synthetic TCI generation",
      MODULE_STATUS["fusion"] in ("TRAINED", "SKIPPED"),
      "fusion status: " + str(MODULE_STATUS["fusion"]) +
      (" (real event dataset: " + str(FUSION_DATASET["path"]) + ")" if FUSION_DATASET else
       " (no real event dataset - nothing manufactured)"))

# 4/5. training + validation
trained = [m for m, s in MODULE_STATUS.items() if s == "TRAINED"]
check("Models trained", len(trained) >= 4, ", ".join(trained))
check("Validation completed", all(m in RESULTS for m in trained), f"{len(RESULTS)} metric sets")

# 6/7. files + metadata
model_files = {m: p for m, p in expected_models.items() if MODULE_STATUS.get(m) == "TRAINED"}
check("Model files saved", all(p.exists() for p in model_files.values()),
      f"{sum(1 for p in model_files.values() if p.exists())}/{len(model_files)}")
check("Metadata saved",
      all(p.with_name(p.stem + '_metadata.json').exists() for p in model_files.values()))

# 8. old models untouched
check("Old models untouched", not changed and not missing_now,
      f"{len(OLD_MODEL_SNAPSHOT)} files verified")

# 9. new names unique
collisions = [p.name for p in model_files.values() if (OLD_MODELS_DIR / p.name).exists()]
check("New model names unique", not collisions, ", ".join(collisions) if collisions else "no collisions")

# 10. preprocessing documented
check("Production preprocessing documented", contract_path.exists(), str(contract_path))

# 11. every sample traceable
manifests = list(V2_RUNS_DIR.rglob("sample_manifest_*.csv"))
check("Every training sample traceable to a real file", len(manifests) > 0,
      f"{len(manifests)} manifest files")

print("=" * 70)
print("FINAL VERIFICATION")
print("=" * 70)
for label, ok, detail in checks:
    mark = "OK" if ok else "!!"
    print(f" [{mark}] {label:48} {detail}")

failed = [c for c in checks if not c[1]]
print()
if failed:
    print(f"{len(failed)} check(s) need attention before promoting any V2 model.")
else:
    print("ALL CHECKS PASSED.")

print()
print("=" * 70)
print("ARTEFACTS")
print("=" * 70)
print("Models   :", V3_MODELS_DIR)
print("Runs     :", V3_RUNS_DIR)
print("Report   :", V3_RUNS_DIR / "EVALUATION_REPORT.md")
print("Compare  :", V3_RUNS_DIR / "MODEL_COMPARISON.md")
print("Index    :", V3_MODELS_DIR / "MODEL_INDEX.json")
print("Contract :", contract_path)
print()
print("Next step: test the V3 models inside the SENTRIX application, then promote")
print("only the ones that genuinely improve real-world behaviour.")

(V2_RUNS_DIR / f"final_verification_{RUN_ID}.json").write_text(
    json.dumps([{"check": c[0], "passed": c[1], "detail": c[2]} for c in checks],
               indent=2), encoding="utf-8")

FINAL VERIFICATION
 [OK] All real datasets found                          G:\Capstone\data
 [OK] Anomaly delegated to the event notebook          SENTRIX_UCF_CRIME_DVS_TRAINING.ipynb | model present: True
 [OK] No synthetic dataset classes detected            notebook file not found on disk
 [OK] No random sample generation calls                no array/tensor generators used to create samples
 [OK] No synthetic TCI generation                      fusion status: SKIPPED (no real event dataset - nothing manufactured)
 [OK] Models trained                                   violence, audio, weapon, fire_smoke
 [OK] Validation completed                             4 metric sets
 [OK] Model files saved                                4/4
 [OK] Metadata saved                                   
 [OK] Old models untouched                             7 files verified
 [OK] New model names unique                           no collisions
 [OK] Production preprocessing documented              G:\Sent

1566

---
---
# APPENDIX A — Final directory structure

```text
G:\Sentrix
│
├── backend
│   └── models
│       ├── violence_classifier.pt              <- OLD (untouched)
│       ├── anomaly_classifier.pt               <- OLD (untouched)
│       ├── audio_classifier.pt                 <- OLD (untouched)
│       ├── weapon_detector.pt                  <- OLD (untouched)
│       ├── fire_smoke_detector.pt              <- OLD (untouched)
│       ├── ...existing models...
│       │
│       ├── v2_real
│       │   └── ucf_crime_dvs                   <- NOTEBOOK 1 (event anomaly)
│       │       ├── ucf_crime_dvs_anomaly_v1.pt
│       │       └── ucf_crime_dvs_anomaly_v1_metadata.json
│       │
│       └── v3_real                             <- THIS NOTEBOOK
│           ├── violence_classifier_real_v3.pt
│           ├── violence_classifier_real_v3_metadata.json
│           ├── audio_classifier_real_v3.pt
│           ├── audio_classifier_real_v3_metadata.json
│           ├── weapon_detector_real_v3.pt
│           ├── weapon_detector_real_v3_metadata.json
│           ├── fire_smoke_detector_real_v3.pt
│           ├── fire_smoke_detector_real_v3_metadata.json
│           ├── tci_xgboost_real_v3.json
│           ├── tci_xgboost_real_v3_metadata.json
│           ├── MODEL_INDEX.json
│           └── PRODUCTION_PREPROCESSING_CONTRACT.json
│
└── training
    ├── SENTRIX_UCF_CRIME_DVS_TRAINING.ipynb    <- NOTEBOOK 1
    ├── SENTRIX_REAL_DATA_RETRAINING_V3.ipynb   <- NOTEBOOK 2 (this one)
    ├── ucf_crime_dvs/                          (official split + gt files)
    ├── cache_ucf_crime_dvs/                    (event tensor cache)
    ├── runs_ucf_crime_dvs/
    └── runs_v3_real
        ├── violence      (history.csv, metrics.json, sample_manifest_*.csv, best_checkpoint.pt)
        ├── audio
        ├── weapon        (train/, val/, data_v2_real.yaml)
        ├── fire_smoke
        ├── fusion
        ├── EVALUATION_REPORT.md
        └── MODEL_COMPARISON.md
```

Datasets stay completely separate and are never written to:

```text
G:\Capstone\data                       G:\event_frame_duration533326
├── violence_data                       └── event_frame_duration533326
├── audio_data\audioset_clips               ├── Abuse ... Vandalism  (.npz)
├── fire_smoke_data                          └── 116 GB - NEVER copied
└── weapon_data
```

## The one non-negotiable rule

**Never sacrifice real-data integrity just to get a high metric.**
A V3 model is promoted into `backend/models/` only after it beats the old model on
realistic validation **and** in application testing.

---
---
# APPENDIX B — Anaconda environment with accelerated PyTorch

Run these in **Anaconda Prompt** (Windows). Do not run them inside this notebook.

## B.0 — Check your GPU driver first

```bat
nvidia-smi
```

The top-right "CUDA Version" is the **maximum** CUDA runtime your driver supports.
Pick a PyTorch wheel whose CUDA build is **less than or equal to** that number:

| Driver reports | Use wheel index |
|---|---|
| 12.6 – 12.7 | `https://download.pytorch.org/whl/cu126` |
| 12.8+ | `https://download.pytorch.org/whl/cu128` |
| 13.x | `https://download.pytorch.org/whl/cu129` (or the newest listed on pytorch.org) |
| no NVIDIA GPU | `https://download.pytorch.org/whl/cpu` |

If unsure, open <https://pytorch.org/get-started/locally/> and copy the command the selector gives you —
the rest of this appendix stays the same.

## B.1 — Create and activate the environment

```bat
conda deactivate
conda create -n sentrix_v2 python=3.11 -y
conda activate sentrix_v2

python -m pip install --upgrade pip setuptools wheel
```

> Python 3.11 is the sweet spot: every dependency below ships prebuilt wheels for it.

## B.2 — Install accelerated PyTorch (pip wheels — the officially supported path)

```bat
:: CUDA 12.8 build (change cu128 to match the table above)
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
```

CPU-only fallback:

```bat
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
```

**Verify immediately — do not skip this:**

```bat
python -c "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"
```

If `torch.cuda.is_available()` prints `False` on an NVIDIA machine, you installed the CPU wheel.
Fix it with:

```bat
pip uninstall -y torch torchvision torchaudio
pip cache purge
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
```

## B.3 — Install the rest of the SENTRIX stack

```bat
pip install ultralytics
pip install xgboost scikit-learn pandas numpy
pip install opencv-python pillow
pip install librosa soundfile audioread
pip install matplotlib seaborn tqdm pyyaml
pip install jupyterlab notebook ipykernel ipywidgets
```

`ultralytics` pulls in its own `torch` requirement — install it **after** PyTorch (as above) so it
does not pull a CPU build over your CUDA build. Re-run the verification in B.2 afterwards.

## B.4 — Register the kernel and launch

```bat
python -m ipykernel install --user --name sentrix_v2 --display-name "Python (sentrix_v2)"

G:
cd G:\Sentrix\training
jupyter lab
```

Open `SENTRIX_REAL_DATA_RETRAINING_V3.ipynb` and select the **Python (sentrix_v2)** kernel
(top-right kernel picker) before running anything.

## B.5 — One-shot copy/paste block

```bat
conda deactivate
conda create -n sentrix_v2 python=3.11 -y
conda activate sentrix_v2
python -m pip install --upgrade pip setuptools wheel
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
pip install ultralytics xgboost scikit-learn pandas numpy opencv-python pillow librosa soundfile audioread matplotlib seaborn tqdm pyyaml jupyterlab notebook ipykernel ipywidgets
python -m ipykernel install --user --name sentrix_v2 --display-name "Python (sentrix_v2)"
python -c "import torch, torchvision, torchaudio, ultralytics, xgboost, librosa; print('torch', torch.__version__, '| cuda', torch.version.cuda, '| available', torch.cuda.is_available())"
```

## B.6 — Reproducible export

```bat
conda activate sentrix_v2
pip freeze > G:\Sentrix\training\requirements_sentrix_v2.txt
conda env export --no-builds > G:\Sentrix\training\environment_sentrix_v2.yml
```

Recreate later with:

```bat
conda env create -f G:\Sentrix\training\environment_sentrix_v2.yml
```

## B.7 — GPU memory notes

- `WEAPON_BATCH_SIZE` / `FIRE_SMOKE_BATCH_SIZE` = 16 at `imgsz=640` needs roughly **6 GB** of VRAM
  with `yolov8n`. Drop to 8 on a 4 GB card; raise to 32 on 12 GB+.
- The event/anomaly notebook needs one extra package: `pip install spikingjelly`.
- `yolov8s.pt` / `yolov8m.pt` give better mAP for more VRAM — set `WEAPON_BASE_WEIGHTS` in CELL 22.
- Mixed precision is enabled automatically when CUDA is present (`USE_AMP` in CELL 4).
- On Windows keep `NUM_WORKERS = 0` inside notebooks; multiprocessing DataLoaders misbehave there.
- If you hit `CUDA out of memory`, restart the kernel, halve the batch size, and rerun that module only.

## B.8 — Non-NVIDIA hardware

```bat
:: Apple Silicon (macOS) - MPS is picked up automatically by CELL 4
pip install torch torchvision torchaudio

:: AMD on Linux (ROCm)
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/rocm6.2
```

Windows + AMD has no official accelerated PyTorch build; use the CPU wheel and expect the
YOLO modules to be slow.